# **Data Preparation for Fine-Tuning - DDIs**

## **Data Preprocessing**

### **1. Identification of Redundant and Bidirectional Entries**

Identification of instances where a specific drug pair appears multiple times. This includes both exact duplicates and "swapped" entries where the order of the drugs is reversed (e.g., Drug A + Drug B vs. Drug B + Drug A).

Outputs a list of these repetitions, showing exactly how they appear in the raw data (e.g., "Omeprazole + Clopidogrel (major) and Clopidogrel + Omeprazole (moderate)") to highlight contradictions or redundancies.

In [ ]:
import json
import os

FILE_PATH = os.path.join("..", "..", "Data", "150_mais_DDI_Dataset_Severity_Mechanisms_and_Alternative", "DDI_Database.json")

def list_simple_repetitions(file_path):
    if not os.path.exists(file_path):
        print("Error: File not found.")
        return

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Dictionary to group pairs regardless of order
    pair_tracker = {}

    for category in ["major", "moderate", "minor"]:
        items = data.get("drug_interactions", {}).get(category, [])
        for item in items:
            d1 = item.get('drug_a', '').strip()
            d2 = item.get('drug_b', '').strip()
            
            if not d1 or not d2: continue

            # Create a key that ignores order (A, B) == (B, A)
            sorted_key = tuple(sorted([d1.lower(), d2.lower()]))
            
            # Record how it appears in the JSON (e.g., "Omeprazole + Clopidogrel")
            entry_str = f"{d1} + {d2} ({category})"
            
            if sorted_key not in pair_tracker:
                pair_tracker[sorted_key] = []
            pair_tracker[sorted_key].append(entry_str)

    print("--- REPETITIONS FOUND ---")
    count = 0
    for key, entries in pair_tracker.items():
        # Only print if there is more than 1 entry for this pair
        if len(entries) > 1:
            count += 1
            drug_names = f"{key[0].title()}, {key[1].title()}"
            variations = " and ".join(entries)
            print(f"{drug_names} appears = {variations}")
            print("-" * 20)
            
    if count == 0:
        print("No repetitions found.")

list_simple_repetitions(FILE_PATH)

--- REPETITIONS FOUND ---
Clopidogrel, Omeprazole appears = Clopidogrel + Omeprazole (major) and Omeprazole + Clopidogrel (moderate)
--------------------
Ciprofloxacin, Theophylline appears = Theophylline + Ciprofloxacin (major) and Ciprofloxacin + Theophylline (moderate)
--------------------
Nitroglycerin, Sildenafil appears = Sildenafil + Nitroglycerin (major) and Nitroglycerin + Sildenafil (minor)
--------------------
Lamotrigine, Valproic Acid appears = Valproic Acid + Lamotrigine (major) and Lamotrigine + Valproic Acid (moderate)
--------------------
Carbamazepine, Risperidone appears = Risperidone + Carbamazepine (major) and Risperidone + Carbamazepine (minor)
--------------------
Levodopa, Metoclopramide appears = Levodopa + Metoclopramide (major) and Metoclopramide + Levodopa (minor)
--------------------
Rifampin, Verapamil appears = Rifampin + Verapamil (major) and Rifampin + Verapamil (moderate)
--------------------
Cephalexin, Metformin appears = Metformin + Cephalexin (mode

#### **1.1 Severity-Based Cleaning**
The raw dataset contains inconsistencies where the same drug pair appears in multiple severity categories (e.g., Sildenafil + Nitroglycerin appearing in both "Major" and "Minor"). 

The following script:
- Defines a priority order: Major > Moderate > Minor
- Iterates through the dataset following this order.
- identifying each unique drug pair (treating Drug A + Drug B as identical to Drug B + Drug A).
- Retains only the first instance encountered (which corresponds to the highest severity level due to the iteration order).
- Discards any subsequent lower-severity duplicates to prevent conflicting labels.
- Saves the consistent, conflict-free data to DDI_Database_Clean.json.

In [ ]:
import json
import os

INPUT_FILE = os.path.join("..", "..", "Data", "150_mais_DDI_Dataset_Severity_Mechanisms_and_Alternative", "DDI_Database.json")
OUTPUT_DIR = "intermediate_data" 
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "DDI_Database_Clean.json")

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

def clean_by_severity(input_path, output_path):
    print(f"--- CLEANING DATASET (Priority: Major > Moderate > Minor) ---")
    
    if not os.path.exists(input_path):
        print(f"Error: File not found at {input_path}")
        return

    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    root = data.get("drug_interactions", {})
    
    # Define Priority Order (Highest to Lowest)
    priority_order = ["major", "moderate", "minor"]
    
    # Dictionary to store the single best entry for each pair
    kept_entries = {}
    
    stats = {
        "processed": 0,
        "kept": 0,
        "discarded": 0
    }

    # Iterate and Filter
    for severity in priority_order:
        items = root.get(severity, [])
        for item in items:
            stats["processed"] += 1
            
            d1 = item.get('drug_a', '').strip()
            d2 = item.get('drug_b', '').strip()
            
            if not d1 or not d2: continue

            # Create a unique key ignoring order
            pair_key = tuple(sorted([d1.lower(), d2.lower()]))
            
            # Decision Logic
            if pair_key not in kept_entries:
                # If haven't seen this pair yet, keep it.
                kept_entries[pair_key] = {
                    "data": item,
                    "severity": severity
                }
                stats["kept"] += 1
            else:
                # If have seen it, it means a higher (or equal) severity version was already processed. Discard this one.
                existing_severity = kept_entries[pair_key]["severity"]
                print(f"Discarding conflict: {d1} + {d2} ({severity}) - Kept existing ({existing_severity})")
                stats["discarded"] += 1

    # Reconstruct JSON Structure
    clean_structure = {
        "drug_interactions": {
            "major": [],
            "moderate": [],
            "minor": []
        }
    }
    
    # Sort entries back into their categories
    for entry in kept_entries.values():
        sev = entry["severity"]
        item_data = entry["data"]
        clean_structure["drug_interactions"][sev].append(item_data)

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(clean_structure, f, indent=4)

    print("-" * 30)
    print(f"Total entries processed: {stats['processed']}")
    print(f"Unique pairs kept:       {stats['kept']}")
    print(f"Duplicates discarded:    {stats['discarded']}")
    print(f"Clean dataset saved to:  {output_path}")

clean_by_severity(INPUT_FILE, OUTPUT_FILE)

--- CLEANING DATASET (Priority: Major > Moderate > Minor) ---
Discarding conflict: Omeprazole + Clopidogrel (moderate) - Kept existing (major)
Discarding conflict: Lamotrigine + Valproic Acid (moderate) - Kept existing (major)
Discarding conflict: Ciprofloxacin + Theophylline (moderate) - Kept existing (major)
Discarding conflict: Rifampin + Verapamil (moderate) - Kept existing (major)
Discarding conflict: Cephalexin + Metformin (minor) - Kept existing (moderate)
Discarding conflict: Metoclopramide + Levodopa (minor) - Kept existing (major)
Discarding conflict: Risperidone + Carbamazepine (minor) - Kept existing (major)
Discarding conflict: Nitroglycerin + Sildenafil (minor) - Kept existing (major)
------------------------------
Total entries processed: 180
Unique pairs kept:       172
Duplicates discarded:    8
Clean dataset saved to:  intermediate_data\DDI_Database_Clean.json


### **2. Missing Values Analysis**

In [ ]:
import json
import os
import pandas as pd

INPUT_FILE = os.path.join("intermediate_data", "DDI_Database_Clean.json")

def check_missing_values(file_path):
    print(f"--- CHECKING FOR MISSING VALUES: {file_path} ---")
    
    if not os.path.exists(file_path):
        print("Error: File not found.")
        return

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    root = data.get("drug_interactions", {})
    categories = ["major", "moderate", "minor"]
    
    required_fields = [
        "drug_a", "drug_b", "severity", 
        "mechanism", "effect", 
        "Safer_alternative", "rationale", "reference"
    ]
    
    missing_counts = {field: 0 for field in required_fields}
    total_entries = 0
    incomplete_entries = []

    for category in categories:
        items = root.get(category, [])
        for item in items:
            total_entries += 1
            entry_has_missing = False
            missing_in_this_entry = []

            for field in required_fields:
                value = item.get(field)
                
                # Check if value is None, or empty string, or just whitespace
                if value is None or (isinstance(value, str) and not value.strip()):
                    missing_counts[field] += 1
                    entry_has_missing = True
                    missing_in_this_entry.append(field)
            
            if entry_has_missing:
                incomplete_entries.append({
                    "Drug Pair": f"{item.get('drug_a')} + {item.get('drug_b')}",
                    "Category": category,
                    "Missing Fields": ", ".join(missing_in_this_entry)
                })

    print(f"Total Entries Analyzed: {total_entries}")
    
    summary_data = []
    for field, count in missing_counts.items():
        summary_data.append({
            "Field": field,
            "Missing Count": count,
            "Missing %": f"{(count/total_entries)*100:.2f}%"
        })
    
    df_summary = pd.DataFrame(summary_data)
    print("\nMISSING VALUES SUMMARY:")
    display(df_summary)

    if incomplete_entries:
        print(f"\n FOUND {len(incomplete_entries)} INCOMPLETE ENTRIES:")
        df_details = pd.DataFrame(incomplete_entries)
        display(df_details)
    else:
        print("\n No missing values found in any field.")

check_missing_values(INPUT_FILE)

--- CHECKING FOR MISSING VALUES: intermediate_data\DDI_Database_Clean.json ---
Total Entries Analyzed: 172

MISSING VALUES SUMMARY:


,Field,Missing Count,Missing %
0,drug_a,0,0.00%
1,drug_b,0,0.00%
2,severity,0,0.00%
3,mechanism,0,0.00%
4,effect,0,0.00%
5,Safer_alternative,0,0.00%
6,rationale,0,0.00%
7,reference,0,0.00%



 No missing values found in any field.


### **3. Manual Classification of Replacement Targets**

A manual review of the dataset revealed that the Safer_alternative field is context-dependent: in some instances, it replaces drug_a, while in others, it replaces drug_b. Since this distinction is not explicitly labeled in the raw data, we must manually classify each entry to ensure the model learns to suggest the correct clinical substitution.

The following script:
- Loads the cleaned dataset (DDI_Database_Clean.json).
- Iterates through each interaction, displaying the Drug Pair, Effect, and Suggested Alternative.
- Asks you to classify which drug the alternative is replacing:
    - A: The alternative replaces Drug A.
    - B: The alternative replaces Drug B.
    - N: Neither (e.g., the advice is to "monitor" or "adjust dose").
    - Both: The alternative replaces both.
- Saves progress automatically when you quit, allowing you to resume later.

In [ ]:
import json
import os
from IPython.display import clear_output

FILE_PATH = os.path.join("intermediate_data", "DDI_Database_Clean.json")

def manual_tagging_tool(file_path):
    if not os.path.exists(file_path):
        print("Error: File not found.")
        return
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    root = data.get("drug_interactions", {})
    categories = ["major", "moderate", "minor"]
    
    total_tagged = 0
    modified = False

    print("--- STARTING MANUAL CLASSIFICATION ---")
    print("Controls: 'A' (Drug A), 'B' (Drug B), 'N' (Neither/Only monitor), 'Both', 'Q' (Save & Quit)\n")

    try:
        # Iterate through categories and items
        for category in categories:
            items = root.get(category, [])
            
            for i, item in enumerate(items):
                # Skip if already tagged (allows resuming progress)
                if "replacement_target" in item:
                    continue
                
                # Visual Interface
                print("="*60)
                print(f"CATEGORY: {category.upper()} | Item {i+1}/{len(items)}")
                print("-" * 60)
                print(f"DRUG A:  {item.get('drug_a')}")
                print(f"DRUG B:  {item.get('drug_b')}")
                print("-" * 60)
                print(f"EFFECT:      {item.get('effect')}")
                print(f"ALTERNATIVE: {item.get('Safer_alternative')}")
                print(f"RATIONALE:   {item.get('rationale')}")
                print("="*60)
                
                # Input validation loop
                while True:
                    choice = input("Which drug is being replaced? (A / B / N / Both) or Q to quit: ").strip().upper()
                    
                    if choice == 'Q':
                        print("\n Saving progress and exiting...")
                        raise StopIteration # Force exit from nested loops
                    
                    if choice in ['A', 'B', 'N', 'BOTH']:
                        # Update the item in memory
                        item['replacement_target'] = choice
                        modified = True
                        total_tagged += 1
                        clear_output(wait=True) # Clears screen for the next item
                        break
                    else:
                        print("Invalid option. Please choose A, B, N, Both, or Q.")

    except StopIteration:
        pass # Clean exit
    
    # Save Changes
    if modified:
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
        print(f"\n Success! {total_tagged} new entries classified and saved to '{os.path.basename(file_path)}'.")
    else:
        print("\n No changes made.")

    # Check Remaining Work
    remaining = 0
    for cat in categories:
        for item in root.get(cat, []):
            if "replacement_target" not in item:
                remaining += 1
    
    if remaining == 0:
        print("Completed! The dataset is 100% classified.")
    else:
        print(f"There are {remaining} entries left to classify. Run the script again to continue.")

manual_tagging_tool(FILE_PATH)


 Success! 21 new entries classified and saved to 'DDI_Database_Clean.json'.
Completed! The dataset is 100% classified.


#### **3.1. Refinement of Complex 'BOTH' Cases**

Entries previously classified as "BOTH" often contain conditional alternatives (e.g., "Levofloxacin or Baclofen"), where one option replaces Drug A and the other replaces Drug B. This script identifies these composite entries, splits the text by "or", and prompts for a granular classification to define exactly which specific option corresponds to which drug, enabling precise generation of substitution advice.

In [ ]:
import json
import os
from IPython.display import clear_output

FILE_PATH = os.path.join("intermediate_data", "DDI_Database_Clean.json")

def refine_both_cases(file_path):
    if not os.path.exists(file_path):
        print("Error: File not found.")
        return

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    root = data.get("drug_interactions", {})
    categories = ["major", "moderate", "minor"]
    
    # List of keywords that indicate an ACTION, not a Drug Name
    # If the text contains these, we skip splitting
    action_keywords = ["monitor", "adjust", "avoid", "space", "consider", "continue", "reduce"]

    count = 0
    modified = False

    print("--- REFINING 'BOTH' CASES (Excluding Actions) ---")
    print("Looking for drug choices (e.g., 'Drug X or Drug Y')...\n")

    try:
        for category in categories:
            items = root.get(category, [])
            
            for item in items:
                target = item.get("replacement_target")
                alt_text = item.get("Safer_alternative", "")
                alt_lower = alt_text.lower()

                # Must be BOTH and have " or "
                if target == "BOTH" and " or " in alt_lower:
                    
                    # FILTER: Check if it's an action phrase (Monitor, Adjust, etc.)
                    # If it contains any action keyword, we SKIP it.
                    is_action = any(keyword in alt_lower for keyword in action_keywords)
                    if is_action:
                        continue

                    # Skip if already processed
                    if "detailed_replacement" in item:
                        continue
                    
                    # Attempt to split
                    parts = alt_text.split(" or ")
                    
                    if len(parts) < 2:
                        continue

                    # --- INTERFACE ---
                    print("="*60)
                    print(f"DRUG A: {item.get('drug_a')}")
                    print(f"DRUG B: {item.get('drug_b')}")
                    print(f"FULL ALTERNATIVE: '{alt_text}'")
                    print("-" * 60)
                    
                    mapping = {}
                    
                    for part in parts:
                        part = part.strip()
                        while True:
                            ans = input(f"Does option '{part}' replace A or B? (A/B/N): ").upper().strip()
                            if ans in ['A', 'B', 'N']:
                                mapping[part] = ans
                                break
                            elif ans == 'Q':
                                raise StopIteration
                    
                    item["detailed_replacement"] = mapping
                    modified = True
                    count += 1
                    clear_output(wait=True)

    except StopIteration:
        print("Saving and exiting...")

    if modified:
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
        print(f"\n Process complete. {count} drug choices detailed.")
    else:
        print("\nNo changes needed or made.")

refine_both_cases(FILE_PATH)


 Process complete. 2 drug choices detailed.


## **Data Split**

In [ ]:
#pip install scikit-learn

### **1. Splitting the DDI Dataset (80/10/10)**
This script divides the cleaned DDI corpus into Train, Dev, and Test sets using stratified sampling to ensure the proportion of Major, Moderate, and Minor interactions is identical across all splits.

In [1]:
import json
import os
from sklearn.model_selection import train_test_split

INPUT_FILE = os.path.join("intermediate_data", "DDI_Database_Clean.json")
OUTPUT_DIR = "datasets_split"

def split_ddi_dataset():
    print("--- SPLITTING DDI DATASET (80% Train / 10% Dev / 10% Test) ---")
    
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found.")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)

    all_interactions = []
    root = data.get("drug_interactions", {})
    
    for severity in ["major", "moderate", "minor"]:
        for item in root.get(severity, []):
            item['severity_label'] = severity # Preserve severity label in the item
            all_interactions.append(item)

    total = len(all_interactions)
    print(f"Total DDI Entries: {total}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    stratify_labels = [x.get('severity_label', 'unknown') for x in all_interactions]

    # 1. Split Train (80%) vs Temp (20%)
    # Stratify ensures we keep the same ratio of major/moderate/minor in all sets
    #train_test_split only divides in two parts 
    train_data, temp_data = train_test_split(
        all_interactions, 
        test_size=0.2, 
        random_state=42, 
        stratify=stratify_labels
    )
    
    # 2. Split Temp (20%) into Dev (10%) and Test (10%)
    stratify_temp = [x.get('severity_label', 'unknown') for x in temp_data]
    # Splitting the remaining 20% in half gives 10% each of the original total
    dev_data, test_data = train_test_split(
        temp_data, 
        test_size=0.5, 
        random_state=42, 
        stratify=stratify_temp
    )

    def save_split(filename, data_list):
        full_path = os.path.join(OUTPUT_DIR, filename)
        with open(full_path, 'w', encoding='utf-8') as f:
            json.dump(data_list, f, indent=4, ensure_ascii=False)
        print(f"Saved {full_path}: {len(data_list)} entries ({len(data_list)/total:.1%})")

    save_split("ddi_split_TRAIN.json", train_data)
    save_split("ddi_split_DEV.json", dev_data)
    save_split("ddi_split_TEST.json", test_data)

split_ddi_dataset()

--- SPLITTING DDI DATASET (80% Train / 10% Dev / 10% Test) ---
Total DDI Entries: 172
Saved datasets_split\ddi_split_TRAIN.json: 137 entries (79.7%)
Saved datasets_split\ddi_split_DEV.json: 17 entries (9.9%)
Saved datasets_split\ddi_split_TEST.json: 18 entries (10.5%)


### **2. Confirmation of the distribution of severity**

This script was made to confirm the distribution of severity in each dataset, TRAIN, DEV, TEST

In [2]:
import json
import os
import pandas as pd

INPUT_DIR = "datasets_split"
FILES = {
    "TRAIN": "ddi_split_TRAIN.json",
    "DEV":   "ddi_split_DEV.json",
    "TEST":  "ddi_split_TEST.json"
}

def verify_ddi_distribution():
    print(f"--- VERIFYING DISTRIBUTION IN: {INPUT_DIR} ---")
    
    stats_list = []

    for split_name, filename in FILES.items():
        filepath = os.path.join(INPUT_DIR, filename)
        
        if not os.path.exists(filepath):
            print(f"Error: {filename} not found.")
            continue
            
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        # Convert to DataFrame to facilitate the counts
        df = pd.DataFrame(data)
        
        # counts and percentages
        counts = df['severity_label'].value_counts()
        percents = df['severity_label'].value_counts(normalize=True) * 100
        
        print(f"\n {split_name} ({len(df)} total entries):")
        for label in counts.index:
            print(f"   - {label}: {counts[label]} ({percents[label]:.2f}%)")

        for label in percents.index:
            stats_list.append({
                "Severity": label,
                "Split": split_name,
                "Percentage": percents[label]
            })

    print("\n" + "="*40)
    print("COMPARISON TABLE")
    print("="*40)
    
    if stats_list:
        df_stats = pd.DataFrame(stats_list)
        #  Pivot the table so that Train/Dev/Test appear as columns
        pivot_table = df_stats.pivot(index="Severity", columns="Split", values="Percentage")
        print(pivot_table.round(2).astype(str) + "%")
    else:
        print("No data to compare.")

verify_ddi_distribution()

--- VERIFYING DISTRIBUTION IN: datasets_split ---

 TRAIN (137 total entries):
   - major: 50 (36.50%)
   - minor: 49 (35.77%)
   - moderate: 38 (27.74%)

 DEV (17 total entries):
   - major: 6 (35.29%)
   - minor: 6 (35.29%)
   - moderate: 5 (29.41%)

 TEST (18 total entries):
   - major: 7 (38.89%)
   - minor: 6 (33.33%)
   - moderate: 5 (27.78%)

COMPARISON TABLE
Split        DEV    TEST   TRAIN
Severity                        
major     35.29%  38.89%   36.5%
minor     35.29%  33.33%  35.77%
moderate  29.41%  27.78%  27.74%


---

## **QA Generation**

---

##### **Converting JSON Datasets to JSONL**
This script iterates through the `data_finetuning` directory, locates the specified Complete Cases JSON files, and exports them line-by-line into new `.jsonl` files.

In [3]:
def convert_json_to_jsonl(directory, filenames):
    print("--- [START] JSON to JSONL Conversion ---")
    
    if not os.path.exists(directory):
        print(f"[ERROR] Directory '{directory}' not found.")
        return

    for filename in filenames:
        json_path = os.path.join(directory, filename)
        
        # Replace the .json extension with .jsonl
        jsonl_filename = filename.replace(".json", ".jsonl")
        jsonl_path = os.path.join(directory, jsonl_filename)
        
        if not os.path.exists(json_path):
            print(f" [SKIP] Source file '{json_path}' not found.")
            continue
            
        print(f" Processing: {filename}...")
        
        try:
            with open(json_path, 'r', encoding='utf-8') as f_in:
                data = json.load(f_in)
                
            # Write each item as an independent JSON string on a new line
            with open(jsonl_path, 'w', encoding='utf-8') as f_out:
                for item in data:
                    # ensure_ascii=False keeps special characters intact
                    f_out.write(json.dumps(item, ensure_ascii=False) + '\n')
                    
            print(f"   -> Success! Saved {len(data)} lines to: {jsonl_filename}")
            
        except json.JSONDecodeError:
            print(f" [ERROR] '{filename}' is not a valid JSON file.")
        except Exception as e:
            print(f" [ERROR] An unexpected error occurred with '{filename}': {e}")
            
    print("\n--- [DONE] Conversion Complete ---")

---

### **- Complete Cases:**
**All the possible combinations of drugs from Medicines with DDI interactions - 3 medicines** 

#### **1. JSON with all possible complete cases - TRAIN, DEV, TEST**

This script automates the creation of a QA dataset. It uses DDI information to create scenarios using pharmaceutical brand data.

- A scenario is only generated if Medication A treats a condition that Medication B does not (and vice-versa), preventing cases of therapeutic duplication.

- Brand-Level Simulation: Cross-references active ingredients with a comprehensive medicine database to generate all possible combinations of commercial brand names.

- Exhaustive & Grouped Output: Generates every valid combination of brands for each interaction and groups them hierarchically

In [5]:
import json
import os
import random
import re
import itertools

DDI_INPUT_DIR = "datasets_split"
MEDS_INPUT_DIR = os.path.join("..", "Medicines", "datasets_split") 

OUTPUT_DIR = os.path.join("data_generation_output", "complete_cases")

# LIMITER: Set to None to generate ALL possible combinations.
# If set to an integer (e.g., 50), it limits the number of pairs per interaction.
MAX_PAIRS_PER_INTERACTION = None 

SEED = 42

# --- HELPER FUNCTIONS ---

def clean_composition_text(text):
    """
    [HELPER] Data Normalization
    Relevance: The raw data often contains text like "Paracetamol (500mg)". 
    To match ingredients accurately, we need to strip parentheses and lower the case 
    to get just "paracetamol".
    """
    text = re.sub(r'\s*\(.*?\)', '', text)
    clean_text = text.strip().lower()
    
    # Dictionary of known spelling variations or aliases
    DRUG_ALIASES = {
        "amoxycillin": "amoxicillin" #the only found
    }
    
    return DRUG_ALIASES.get(clean_text, clean_text)

def get_composition_str(med_dict):
    """
    [HELPER] String Formatting
    Relevance: Converts the list of ingredients ['A', 'B'] into a readable string "A, B" 
    for the final JSON output.
    """
    comp_list = med_dict.get('Composition', [])
    return ", ".join(comp_list) if comp_list else "Unknown"

def build_medicine_index(meds_data):
    """
    [HELPER] Performance Optimization (Indexing)
    Relevance: Instead of searching through thousands of medicines linearly for every interaction,
    we create a hash map (dictionary).
    - Key: Ingredient Name (e.g., "ibuprofen")
    - Value: List of all commercial medicines containing ONLY that ingredient.
    """
    index = {}
    meds_list = meds_data.values() if isinstance(meds_data, dict) else meds_data
    for med in meds_list:
        comp_list = med.get('Composition', [])
        if len(comp_list) != 1: continue 
        clean_key = clean_composition_text(comp_list[0])
        if clean_key not in index: index[clean_key] = []
        index[clean_key].append(med)
    return index

def extract_drug_candidates(text):
    """
    [HELPER] Text Parsing 
    Relevance: The 'Safer_alternative' field is natural text (e.g., "azithromycin or clarithromycin").
    This function uses Regex to extract the specific drug names into a list: ['azithromycin', 'clarithromycin'].
    It handles parentheses and excludes non-drug words like "monitor".
    """
    candidates = set()
    parts = re.split(r' or |\/|;', text, flags=re.IGNORECASE)
    for part in parts:
        part = part.strip()
        if not part: continue
        
        # Extract content from patterns like "Alternative antibiotic (e.g., Ceftazidime)"
        # Regex: \( = real parenthesis | e.g., = exact text "e.g.," | \s* = optional spaces | (.*?) = THE NAME OF THE DRUG
        eg_match = re.search(r'\(e\.g\.,\s*(.*?)\)', part, re.IGNORECASE) #text in () with e.g.
        parens_match = re.search(r'\((.*?)\)', part) #text in ()
        text_outside = re.sub(r'\(.*?\)', '', part).strip() #text out of ()
        
        if text_outside: candidates.add(text_outside)
        if eg_match: candidates.add(eg_match.group(1).strip())
        elif parens_match:
            inside = parens_match.group(1).strip()
            # Safety filter: ensure we don't extract instructions as drug names
            if not any(x in inside.lower() for x in ["caution", "monitor"]): #e.g: "Morphine or Oxycodone (with caution)"
                candidates.add(inside)
    return list(candidates)

def find_med_c_candidates(alt_text, med_index, target_use):
    """
    [HELPER] Candidate Search (Logic Core)
    Relevance: Finds commercial medicines (Med C) that satisfy TWO conditions:
    1. Chemical Match: Must match the ingredient suggested in 'alt_text'.
    2. Clinical Match: Must be indicated for 'target_use' (the condition we need to treat).
    """
    candidates = []
    potential_drugs = extract_drug_candidates(alt_text)
    for drug in potential_drugs:
        clean = clean_composition_text(drug)
        if clean in med_index:
            for med in med_index[clean]:
                # Check if the candidate med treats the required condition
                if any(target_use.lower() in u.lower() for u in med.get('Uses', [])):
                    candidates.append(med)
    return candidates

def get_unique_uses(source_med, comparison_med):
    """
    [HELPER] 
    Relevance: This is the STRICT filter. It returns a list of uses from 'source_med' 
    that are NOT covered by 'comparison_med'.
    
    Because if Med A treats 'Pain' and Med B also treats 'Pain', proposing Med B for 'Pain' 
    is redundant. We only want to generate scenarios where Med B adds a NEW therapeutic value.
    """
    unique_uses = []
    source_uses = source_med.get('Uses', [])
    # Create a string of all comparison uses for loose matching (case-insensitive)
    comp_uses_str = " ".join(comparison_med.get('Uses', [])).lower()
    
    for use in source_uses:
        # Check if this specific use text appears loosely in the other med's uses
        if use.lower() not in comp_uses_str:
            unique_uses.append(use)
    return unique_uses

def get_valid_alternatives(alternative_string, med_index, condition_to_treat):
    valid_alts = []
    seen_names = set()
    c_cands = find_med_c_candidates(alternative_string, med_index, condition_to_treat)
    for mc in c_cands:
        name = mc['Medicine Name']
        if name not in seen_names:
            valid_alts.append({
                "medicine_name": name,
                "composition": get_composition_str(mc)
            })
            seen_names.add(name)
    return valid_alts, seen_names

# --- MAIN FUNCTION ---

def generate_grouped_dataset(ddi_file, meds_file, out_qa, out_left, out_unused, level_name):
    random.seed(SEED)
    
    ddi_path = os.path.join(DDI_INPUT_DIR, ddi_file)
    meds_path = os.path.join(MEDS_INPUT_DIR, meds_file)
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    print(f"\nGenerating {level_name} (Strict Non-Redundant Logic)...")

    with open(ddi_path, 'r', encoding='utf-8') as f: ddi_list = json.load(f)
    with open(meds_path, 'r', encoding='utf-8') as f: meds_data = json.load(f)
    med_index = build_medicine_index(meds_data)
    
    used_med_names = set()
    grouped_output = []
    ddi_leftovers = [] 
    
    total_cases_count = 0
    both_cases_count = 0  

    # Process each DDI Rule
    for item in ddi_list:
        d1_raw = item.get('drug_a', '').strip()
        d2_raw = item.get('drug_b', '').strip()
        d1_key, d2_key = d1_raw.lower(), d2_raw.lower()
        
        # Retrieve all brands for Drug A and Drug B from index
        meds_a = med_index.get(d1_key)
        meds_b = med_index.get(d2_key)
        
        # Skip if brands are missing (stored in leftovers)
        if not meds_a or not meds_b:
            ddi_leftovers.append(item)
            continue
        
        interaction_group = {
            "interaction_pair": [d1_raw, d2_raw],
            "severity": item.get('severity_label'),
            "interaction_info": {
                "mechanism": item.get('mechanism'),
                "effect": item.get('effect'),
                "recommendation": item.get("Safer_alternative", ""),
                "rationale": item.get('rationale')
            },
            "generated_cases": [] 
        }

        # Generate ALL combinations (Cartesian Product)
        ab_pairs = list(itertools.product(meds_a, meds_b))
        random.shuffle(ab_pairs) # Randomize order for variety, but we process all valid pairs.
        
        count_for_this_group = 0

        for ma, mb in ab_pairs:
            if not ma.get('Uses') or not mb.get('Uses'): continue
            
            # 1. Identify "Unique Uses" for A (What A does that B doesn't)
            valid_uses_for_A = get_unique_uses(ma, mb)
            # 2. Identify "Unique Uses" for B (What B does that A doesn't)
            valid_uses_for_B = get_unique_uses(mb, ma)

            # Check: If either list is empty, it implies full overlap/redundancy.
            # E.g., if A treats [Pain] and B treats [Pain, Inflammation],
            # valid_uses_for_A is empty. Adding B would be redundant for Pain.
            if not valid_uses_for_A or not valid_uses_for_B:
                continue

            # 3. Pick Random Contexts from the Valid Unique Lists
            # This ensures condition_a is something ONLY Med A treats in this pair context.
            # This ensures condition_b is something ONLY Med B treats in this pair context.
            condition_a = random.choice(valid_uses_for_A)
            condition_b = random.choice(valid_uses_for_B)
            
            # --- FIND ALTERNATIVES (Med C) ---
            target = item.get("replacement_target") # "A" or "B"
            alt_text = item.get("Safer_alternative", "")
            detailed_replacement = item.get("detailed_replacement", {})
            
            solutions_data = {}

            if target in ["A", "B"] and alt_text:
                #  LOGIC: The replacement must treat the condition of the drug being REMOVED:
                use_to_match = condition_a if target == "A" else condition_b
                alts, seen_c = get_valid_alternatives(alt_text, med_index, use_to_match)
                
                if alts:
                    used_med_names.update(seen_c)
                    solutions_data = {
                        "target_replaced": target,
                        "condition_treated": use_to_match,
                        "alternative_meds": alts,
                        "count": len(alts)
                    }

            elif target == "BOTH" and detailed_replacement:
                drugs_for_A = [drug for drug, t in detailed_replacement.items() if t == "A"]
                drugs_for_B = [drug for drug, t in detailed_replacement.items() if t == "B"]
                
                alt_str_A = " or ".join(drugs_for_A)
                alt_str_B = " or ".join(drugs_for_B)
                
                alts_A, seen_A = get_valid_alternatives(alt_str_A, med_index, condition_a)
                alts_B, seen_B = get_valid_alternatives(alt_str_B, med_index, condition_b)
                
                if alts_A and alts_B:
                    used_med_names.update(seen_A)
                    used_med_names.update(seen_B)
                    
                    solutions_data = {
                        "target_replaced": "BOTH",
                        "options": [
                            {
                                "target_replaced": "A",
                                "condition_treated": condition_a,
                                "alternative_meds": alts_A,
                                "count": len(alts_A)
                            },
                            {
                                "target_replaced": "B",
                                "condition_treated": condition_b,
                                "alternative_meds": alts_B,
                                "count": len(alts_B)
                            }
                        ]
                    }
                    both_cases_count += 1  

            if not solutions_data:
                continue

            case_entry = {
                "case_id": f"{level_name}_{len(grouped_output)}_{count_for_this_group + 1}",
                "medication_A": {
                    "medicine_name": ma['Medicine Name'],
                    "composition": get_composition_str(ma),
                    "condition": condition_a
                },
                "medication_B": {
                    "medicine_name": mb['Medicine Name'],
                    "composition": get_composition_str(mb),
                    "condition": condition_b
                },
                "solutions": solutions_data
            }
            
            interaction_group["generated_cases"].append(case_entry)
            
            used_med_names.add(ma['Medicine Name'])
            used_med_names.add(mb['Medicine Name'])
            count_for_this_group += 1
            total_cases_count += 1

        # Save group if valid cases exist
        if interaction_group["generated_cases"]:
            grouped_output.append(interaction_group)
        else:
            ddi_leftovers.append(item)

    # --- SAVE FILES ---
    # Calculate unused medicines
    
    all_meds_list = meds_data.values() if isinstance(meds_data, dict) else meds_data
    unused_meds = [m for m in all_meds_list if m.get('Medicine Name') not in used_med_names]

    with open(os.path.join(OUTPUT_DIR, out_qa), 'w', encoding='utf-8') as f:
        json.dump(grouped_output, f, indent=4, ensure_ascii=False)
    with open(os.path.join(OUTPUT_DIR, out_left), 'w', encoding='utf-8') as f:
        json.dump(ddi_leftovers, f, indent=4, ensure_ascii=False)
    with open(os.path.join(OUTPUT_DIR, out_unused), 'w', encoding='utf-8') as f:
        json.dump(unused_meds, f, indent=4, ensure_ascii=False)
    
    print(f" {level_name}: Generated {len(grouped_output)} groups.")
    print(f"   Total Scenarios: {total_cases_count}")
    print(f"   Total 'BOTH' Scenarios: {both_cases_count}") 
    print(f"   Saved to: {OUTPUT_DIR}")

if __name__ == "__main__":
    #generate_grouped_dataset("ddi_split_TRAIN.json", "dataset_meds_TRAIN_v1.json", "complete_cases_TRAIN.json", "DDI_Leftovers_TRAIN.json", "unused_medicines_TRAIN.json", "TRAIN")
    #generate_grouped_dataset("ddi_split_DEV.json", "dataset_meds_DEV_v1.json", "complete_cases_DEV.json", "DDI_Leftovers_DEV.json", "unused_medicines_DEV.json", "DEV")
    generate_grouped_dataset("ddi_split_TRAIN.json", "dataset_meds_TRAIN_v2.json", "complete_cases_TRAIN.json", "DDI_Leftovers_TRAIN.json", "unused_medicines_TRAIN.json", "TRAIN")
    generate_grouped_dataset("ddi_split_DEV.json", "dataset_meds_DEV_v2.json", "complete_cases_DEV.json", "DDI_Leftovers_DEV.json", "unused_medicines_DEV.json", "DEV")
    generate_grouped_dataset("ddi_split_TEST.json", "dataset_meds_TEST_v2.json", "complete_cases_TEST.json", "DDI_Leftovers_TEST.json", "unused_medicines_TEST.json", "TEST")


Generating TRAIN (Strict Non-Redundant Logic)...
 TRAIN: Generated 20 groups.
   Total Scenarios: 4712
   Total 'BOTH' Scenarios: 280
   Saved to: data_generation_output\complete_cases

Generating DEV (Strict Non-Redundant Logic)...
 DEV: Generated 1 groups.
   Total Scenarios: 42
   Total 'BOTH' Scenarios: 0
   Saved to: data_generation_output\complete_cases

Generating TEST (Strict Non-Redundant Logic)...
 TEST: Generated 2 groups.
   Total Scenarios: 44
   Total 'BOTH' Scenarios: 0
   Saved to: data_generation_output\complete_cases


#### **2. QA formatting**

##### **2.1. TRAIN and DEV datasets**

This script processes the `complete_cases` JSON and converts it into the format (`instruction`, `input`, `output`).

- **Clinical Logic:** The script assigns which drug is "current" and which is "proposed" based on the interaction's replacement target. It prioritizes substituting the newly proposed medication rather than unnecessarily disrupting a patient's established current therapy.
- **Scenario Selection:** By automatically sorting the generated cases based on the scoring system, the script extracts only the single *highest-scoring* representative case per interaction rule. 
- **Formatting:** The output separates `**Mechanism:**`, `**Effect:**`, and `**Recommendation:**` into distinct fields. If the source database misclassifies a clinical management action (e.g., "Monitor for toxicity") as a effect, the script purges the `Effect` field and dynamically re-routes the directive into the `Recommendation` field. If an input has `Effect: "Monitor for drowsiness"`, then dynamically morphs into `Recommendation: Monitor for drowsiness. Furthermore, a possible safer alternative is...`

In [6]:
import json
import random
import re
import os

random.seed(42)

INSTRUCTIONS = [
    "Offer clear, professional clinical support to the treating physician. Review the proposed medication regimen for potential drug-drug interactions.",
    "You are an advanced AI medical assistant. Provide concise, professional advice to the attending physician. Evaluate the clinical safety of adding this new medication to the patient's current therapy.",
    "You are a specialized clinical AI. Offer clear, professional advice to the treating physician. Provide clinical decision support regarding the following drug combination.",
    "Act as an expert AI medical assistant. Deliver brief, evidence-based recommendations to the healthcare provider. Assess this patient's medication plan and suggest adjustments if necessary."
]

INPUT_TEMPLATES = [
    "Patient is currently on a regimen including {med_a_comp_clean} [{med_a_brand}] for {cond_a}.\nProposed Action: Initiate {med_b_comp_clean} [{med_b_brand}] for {cond_b}.",
    "Current medication: {med_a_comp_clean} [{med_a_brand}]- Indication: {cond_a}.\nConsidering adding: {med_b_comp_clean} [{med_b_brand}] - Indication: {cond_b}.",
    "A patient taking {med_a_comp_clean} [{med_a_brand}] for {cond_a} requires medication for {cond_b}. I'm considering prescribing {med_b_comp_clean} [{med_b_brand}]."
]

DISCLAIMER = "Clinical note: The significance of drug–drug interactions may vary depending on individual patient factors and specific drug characteristics, including dosage. A thorough clinical evaluation is recommended before any therapeutic adjustments."

# --- REGEX PATTERNS FOR SCORING ---
# Gold Level: Maximum Priority
SYSTEMIC_PATTERN = re.compile(r'\b(tablet|capsule)\b', re.IGNORECASE)
#Avoid if possible
TOPICAL_PATTERN = re.compile(r'\b(cream|gel|ointment|spray|drops?|lotion|shampoo|topical|soap|syrup|suppository)\b', re.IGNORECASE)

def clean_clinical_text(text):
    if not text:
        return ""
    text = text.strip()
    
    text = text[0].upper() + text[1:]
    if not text.endswith('.'):
        text += '.'
        
    return text

def strip_dosage(composition_text):
    return re.sub(r'\s*\(.*?\)', '', composition_text).strip()

def format_bullet_list(meds_list):
    bullets = ""
    for med in meds_list:
        bullets += f"\n- {med['medicine_name']} ({med['composition']})"
    return bullets

def smart_lower(text):
    if not text: 
        return ""
    parts = re.split(r'(\(.*?\))', str(text))
    result = ""
    for part in parts:
        if part.startswith('(') and part.endswith(')'):
            result += part 
        else:
            result += part.lower()  
    return result.strip()

def generate_alpaca_dataset(input_file, output_file, dataset_name):
    print(f"--- [START] Generating Alpaca Dataset for {dataset_name} ---")
    
    with open(input_file, 'r', encoding='utf-8') as f: 
        grouped_data = json.load(f)

    alpaca_dataset = []
    
    # Counters for Stats
    stats_gold = 0
    stats_silver = 0
    stats_bronze = 0

    for group in grouped_data:
        cases = group.get("generated_cases")
        if not cases:
            continue

        severity = group["severity"].upper()
        info = group["interaction_info"]
        
        mechanism = clean_clinical_text(info["mechanism"])
        effect_raw = clean_clinical_text(info["effect"])
        rationale = clean_clinical_text(info["rationale"])
        recommended_active_drug = info["recommendation"] 

        # --- SCORING SYSTEM FOR SELECTION ---
        def score_medication(med_name):
            if TOPICAL_PATTERN.search(med_name):
                return -10 # Heavy penalty for topical/syrup/injection
            elif SYSTEMIC_PATTERN.search(med_name):
                return 2   # Maximum bonus for Tablet/Capsule
            else:
                return 1   # Generic systemic fallback

        def score_case(c):
            a_name = c["medication_A"]["medicine_name"]
            b_name = c["medication_B"]["medicine_name"]
            return score_medication(a_name) + score_medication(b_name)

        # Sort all generated cases from highest score to lowest
        cases.sort(key=score_case, reverse=True)
        
        # The best case will always be at index 0
        case = cases[0]
        
        best_score = score_case(case)
        if best_score == 4:
            stats_gold += 1   # Ideal: Tab + Tab
        elif best_score >= 2:
            stats_silver += 1 # Good: Tab + Inj OR Inj + Inj
        else:
            stats_bronze += 1 # Fallback: At least one Topical involved
        # --------------------------------------------------------------

        med_A = case["medication_A"]
        med_B = case["medication_B"]
        solutions = case["solutions"]
        
        target = solutions["target_replaced"]

        drug_a_clean = strip_dosage(med_A['composition'])
        drug_b_clean = strip_dosage(med_B['composition'])

        instruction = random.choice(INSTRUCTIONS)
        input_template = random.choice(INPUT_TEMPLATES)
        
        if target == "A":
            current_med = med_B
            proposed_med = med_A
        else:
            current_med = med_A
            proposed_med = med_B
            
        cond_current = smart_lower(str(current_med.get('condition', '')))
        cond_proposed = smart_lower(str(proposed_med.get('condition', '')))
        
        input_text = input_template.format(
            med_a_brand=current_med['medicine_name'], 
            med_a_comp_clean=strip_dosage(current_med['composition']), 
            cond_a=cond_current,
            med_b_brand=proposed_med['medicine_name'], 
            med_b_comp_clean=strip_dosage(proposed_med['composition']), 
            cond_b=cond_proposed
        )

        opening_variations = [
            f"Caution: Concurrent use of {drug_a_clean} and {drug_b_clean} carries a {severity} interaction risk.",
            f"Interaction Warning ({severity}): {drug_a_clean} combined with {drug_b_clean}."
        ]
        
        output_lines = [
            random.choice(opening_variations),
            f"**Mechanism:** {mechanism}" 
        ]

        is_monitor_effect = effect_raw.lower().startswith("monitor")

        if not is_monitor_effect and effect_raw:
            output_lines.append(f"**Effect:** {effect_raw}")

        rec_prefix = f"{effect_raw} Alternatively, to avoid this interaction, a safer option" if is_monitor_effect else "A possible safer alternative"

        if target in ["A", "B"]:
            cond_to_treat = smart_lower(str(solutions.get("condition_treated", "")))
            drug_to_replace = drug_a_clean if target == "A" else drug_b_clean
            meds_c_bullets = format_bullet_list(solutions["alternative_meds"])
            
            action_suffix = ""
            if "or adjust doses" in recommended_active_drug.lower():
                recommended_active_drug = re.sub(r'(?i)\s*or adjust doses', '', recommended_active_drug)
                action_suffix = " or adjust doses"

            output_lines.append(f"\n**Recommendation:** {rec_prefix} is to replace {drug_to_replace} with {recommended_active_drug} for {cond_to_treat}{action_suffix}.")
            output_lines.append(f"\n**Possible commercial substitutes:**")
            output_lines.append(meds_c_bullets)
        
        elif target == "BOTH":
            opt_A = solutions["options"][0]
            opt_B = solutions["options"][1]
            
            intro_text = f"{effect_raw} Furthermore, there" if is_monitor_effect else "There"
            output_lines.append(f"\n**Recommendation:** {intro_text} are two options to adjust the therapy and avoid this interaction:")

            cond_A_lower = smart_lower(str(opt_A.get('condition_treated', '')))
            recommended_drug_A = strip_dosage(opt_A["alternative_meds"][0]["composition"])
            
            output_lines.append(f"\nOption 1: For {cond_A_lower}, replace {drug_a_clean} with {recommended_drug_A}.")
            output_lines.append(f"Possible commercial substitutes:")
            output_lines.append(format_bullet_list(opt_A["alternative_meds"]))
            
            cond_B_lower = smart_lower(str(opt_B.get('condition_treated', '')))
            recommended_drug_B = strip_dosage(opt_B["alternative_meds"][0]["composition"])
            
            output_lines.append(f"\nOption 2: For {cond_B_lower}, replace {drug_b_clean} with {recommended_drug_B}.")
            output_lines.append(f"**Possible commercial substitutes:**")
            output_lines.append(format_bullet_list(opt_B["alternative_meds"]))

        output_lines.append(f"\n**Rationale:** {rationale}")
        output_lines.append(f"\n*{DISCLAIMER}*")

        output_text = "\n".join(output_lines)

        alpaca_dataset.append({
            "instruction": instruction,
            "input": input_text,
            "output": output_text
        })

    with open(output_file, 'w', encoding='utf-8') as f: 
        json.dump(alpaca_dataset, f, indent=4, ensure_ascii=False)

    print(f"Success! Generated {len(alpaca_dataset)} dynamic QA pairs for {dataset_name}.")
    print(f" -> Quality Stats: {stats_gold} Ideal (Tab+Tab) | {stats_silver} Good (Systemic) | {stats_bronze} Fallback (Topical)")
    print(f"File saved to: {output_file}\n")


if __name__ == "__main__":
    os.makedirs("data_finetuning", exist_ok=True)

    generate_alpaca_dataset(
        input_file="data_generation_output/complete_cases/complete_cases_TRAIN.json",
        output_file="data_finetuning/complete_cases_TRAIN.json",
        dataset_name="TRAIN"
    )
    
    generate_alpaca_dataset(
        input_file="data_generation_output/complete_cases/complete_cases_DEV.json",
        output_file="data_finetuning/complete_cases_DEV.json",
        dataset_name="DEV"
    )

--- [START] Generating Alpaca Dataset for TRAIN ---
Success! Generated 20 dynamic QA pairs for TRAIN.
 -> Quality Stats: 19 Ideal (Tab+Tab) | 1 Good (Systemic) | 0 Fallback (Topical)
File saved to: data_finetuning/complete_cases_TRAIN.json

--- [START] Generating Alpaca Dataset for DEV ---
Success! Generated 1 dynamic QA pairs for DEV.
 -> Quality Stats: 1 Ideal (Tab+Tab) | 0 Good (Systemic) | 0 Fallback (Topical)
File saved to: data_finetuning/complete_cases_DEV.json



##### **2.2. TEST dataset**

This script processes `complete_cases_TEST.json` to generate an evaluation dataset.
The generated QA pairs are partitioned into two files:
* **Open-Ended QA (`_open.json`)** 
* **Multiple Choice Questions (`_mcq.json`)** 

**Key Logic**
* **Scenario Selection:** Uses a scoring algorithm to prioritize systemic interactions. Solid oral dose forms (tablets/capsules) receive maximum priority (+2 points), while localized forms (e.g., creams, gels) are penalized (-10 points). The top 2 highest-scoring distinct cases are extracted for testing.
* **Clinical Logic:** Assign medications as "current" (therapy) or "proposed" (new addition). This forces the model to reject the new addition rather than swapping a current therapy.
* **Distractor Pools (MCQs):**
    * **Task 2 (Severity Classification)** 
    * **Task 3 (Substitution):** Incorrect drug options are filtered from the global database using a **bidirectional semantic check** (ensuring distractors do not treat the patient's target condition) and a chemical composition filter.

In [18]:
import json
import random
import re
import os

random.seed(42)


INPUT_FILE = os.path.join("data_generation_output", "complete_cases", "complete_cases_TEST.json")
MEDS_FILE = os.path.join("..", "Medicines", "datasets_split", "dataset_meds_TEST_v2.json")
DDI_FILE = os.path.join("datasets_split", "ddi_split_TEST.json") 
OUTPUT_DIR = "data_finetuning"

# --- REGEX PATTERNS FOR SCORING ---
# Gold Level: Maximum Priority
SYSTEMIC_PATTERN = re.compile(r'\b(tablet|capsule)\b', re.IGNORECASE)
#Avoid if possible
TOPICAL_PATTERN = re.compile(r'\b(cream|gel|ointment|spray|drops?|lotion|shampoo|topical|soap|syrup|suppository)\b', re.IGNORECASE)

INSTRUCTIONS_OPEN_QA = [
    "Offer clear, professional clinical support to the treating physician. Review the proposed medication regimen for potential drug-drug interactions.",
    "You are an advanced AI medical assistant. Provide concise, professional advice to the attending physician. Evaluate the clinical safety of adding this new medication to the patient's current therapy.",
    "You are a specialized clinical AI. Offer clear, professional advice to the treating physician. Provide clinical decision support regarding the following drug combination.",
    "Act as an expert AI medical assistant. Deliver brief, evidence-based recommendations to the healthcare provider. Assess this patient's medication plan and suggest adjustments if necessary."
]

INPUT_TEMPLATES_OPEN_QA = [
    "Current medication: {med_a_comp_clean} [{med_a_brand}]- Indication: {cond_a}.\nConsidering adding: {med_b_comp_clean} [{med_b_brand}] - Indication: {cond_b}.",
    "A patient taking {med_a_comp_clean} [{med_a_brand}] for {cond_a} requires medication for {cond_b}. I'm considering prescribing {med_b_comp_clean} [{med_b_brand}]."
]

INSTRUCTION_MCQ = "Read the question and options carefully. Output ONLY the letter (A, B, C, or D) corresponding to the correct answer."

DISCLAIMER = "Clinical note: The significance of drug–drug interactions may vary depending on individual patient factors and specific drug characteristics, including dosage. A thorough clinical evaluation is recommended before any therapeutic adjustments."


# --- FORMATTING HELPERS ---

def clean_clinical_text(text):
    if not text:
        return ""
    text = text.strip()
    
    text = text[0].upper() + text[1:]
    if not text.endswith('.'):
        text += '.'
        
    return text

def format_bullet_list(meds_list):
    bullets = ""
    for med in meds_list[:4]: 
        bullets += f"\n- {med['medicine_name']} ({med['composition']})"
    return bullets

def strip_dosage(composition_text):
    return re.sub(r'\s*\(.*?\)', '', composition_text).strip()

def get_composition_str(med_dict):
    comp_list = med_dict.get('Composition', [])
    return ", ".join(comp_list) if comp_list else "Unknown"

def smart_lower(text):
    if not text: 
        return ""
    parts = re.split(r'(\(.*?\))', str(text))
    result = ""
    for part in parts:
        if part.startswith('(') and part.endswith(')'):
            result += part 
        else:
            result += part.lower()  
    return result.strip()

# --- MAIN FUNCTION ---

def generate_test_qa_dataset():
    print("--- [START] Generating Evaluation Dataset (Complete Cases TEST) ---")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if not os.path.exists(INPUT_FILE) or not os.path.exists(MEDS_FILE) or not os.path.exists(DDI_FILE):
        print(" [ERROR] Required files not found.")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        grouped_data = json.load(f)
        
    with open(MEDS_FILE, 'r', encoding='utf-8') as f:
        meds_data = json.load(f)
        
    with open(DDI_FILE, 'r', encoding='utf-8') as f:
        ddi_raw_data = json.load(f)
        
    all_meds_list = list(meds_data.values()) if isinstance(meds_data, dict) else meds_data

    test_dataset_open = []
    test_dataset_mcq = []
    skipped_count = 0
    
    # Counters for Stats (We track the lowest score among the 2 cases used)
    stats_gold = 0
    stats_silver = 0
    stats_bronze = 0
    
    for group in grouped_data:
        severity = group.get("severity", "Unknown").upper()
        info = group["interaction_info"]
        
        mechanism = clean_clinical_text(info["mechanism"])
        effect = clean_clinical_text(info["effect"])
        rationale = clean_clinical_text(info["rationale"])
        recommended_active_drug = info.get("recommendation", "")

        cases = group.get("generated_cases", [])
        
        if len(cases) < 2:
            skipped_count += 1
            continue 

        # --- SCORING SYSTEM FOR SELECTION ---
        def score_medication(med_name):
            if TOPICAL_PATTERN.search(med_name):
                return -10 # Heavy penalty for topical/syrup/injection
            elif SYSTEMIC_PATTERN.search(med_name):
                return 2   # Maximum bonus for Tablet/Capsule
            else:
                return 1   # Generic systemic fallback

        def score_case(c):
            a_name = c["medication_A"]["medicine_name"]
            b_name = c["medication_B"]["medicine_name"]
            return score_medication(a_name) + score_medication(b_name)

        # Sort all generated cases from highest score to lowest
        cases.sort(key=score_case, reverse=True)
        
        # Select the top 2 highest scoring cases 
        case_1 = cases[0] 
        case_2 = cases[1] 
        
        weakest_score = min(score_case(case_1), score_case(case_2))
        
        if weakest_score == 4:
            stats_gold += 1   # Both cases are Ideal: Tab + Tab
        elif weakest_score >= 2:
            stats_silver += 1 # Both cases are Good: Systemic
        else:
            stats_bronze += 1 # At least one case involves a Topical/Fallback
        # --------------------------------------------------------------------------

        # =========================================================
        # TASK 1: THE ACTION TEST (Open-Ended QA) -> Uses Case 1
        # =========================================================

        med_A_1 = case_1["medication_A"]
        med_B_1 = case_1["medication_B"]
        target_1 = case_1["solutions"]["target_replaced"]
        
        if target_1 == "A":
            current_med_1 = med_B_1
            proposed_med_1 = med_A_1
        else:
            current_med_1 = med_A_1
            proposed_med_1 = med_B_1

        cond_current_1 = smart_lower(str(current_med_1.get('condition', '')))
        cond_proposed_1 = smart_lower(str(proposed_med_1.get('condition', '')))
        
        drug_curr_clean_1 = strip_dosage(current_med_1['composition'])
        drug_prop_clean_1 = strip_dosage(proposed_med_1['composition'])

        input_template = random.choice(INPUT_TEMPLATES_OPEN_QA)
        input_text_1 = input_template.format(
            med_a_brand=current_med_1['medicine_name'], med_a_comp_clean=drug_curr_clean_1, cond_a=cond_current_1,
            med_b_brand=proposed_med_1['medicine_name'], med_b_comp_clean=drug_prop_clean_1, cond_b=cond_proposed_1
        )
        
        cond_to_treat_1 = smart_lower(case_1["solutions"]["condition_treated"])
        
        drug_a_clean_1 = strip_dosage(med_A_1['composition'])
        drug_b_clean_1 = strip_dosage(med_B_1['composition'])
        drug_to_replace_1 = drug_a_clean_1 if target_1 == "A" else drug_b_clean_1
        meds_c_bullets_1 = format_bullet_list(case_1["solutions"]["alternative_meds"])
        
        opening_variations = [
            f"Caution: Concurrent use of {drug_a_clean_1} and {drug_b_clean_1} carries a {severity} interaction risk.",
            f"Interaction Warning ({severity}): {drug_a_clean_1} combined with {drug_b_clean_1}."
        ]
        
        output_lines_1 = [
            random.choice(opening_variations),
            f"**Mechanism:** {mechanism}",
            f"**Effect:** {effect}",
            f"\n**Recommendation:** A possible safer alternative is to replace {drug_to_replace_1} with {recommended_active_drug} for {cond_to_treat_1}.",
            f"\n**Possible commercial substitutes:**",
            f"{meds_c_bullets_1}", 
            f"\n**Rationale:** {rationale}",
            f"\n*{DISCLAIMER}*"
        ]
        test_dataset_open.append({
            "instruction": random.choice(INSTRUCTIONS_OPEN_QA),
            "input": input_text_1,
            "output": "\n".join(output_lines_1)
        })

        # =========================================================
        # TASK 2 & 3 SETUP -> Uses Case 2
        # =========================================================
        med_A_2 = case_2["medication_A"]
        med_B_2 = case_2["medication_B"]
        sols_2 = case_2["solutions"]
        target_2 = sols_2["target_replaced"]

        if target_2 == "A":
            current_med_2 = med_B_2
            proposed_med_2 = med_A_2
        else:
            current_med_2 = med_A_2
            proposed_med_2 = med_B_2

        cond_curr_2 = smart_lower(str(current_med_2.get('condition', '')))
        cond_prop_2 = smart_lower(str(proposed_med_2.get('condition', '')))
        
        drug_curr_clean_2 = strip_dosage(current_med_2['composition'])
        drug_prop_clean_2 = strip_dosage(proposed_med_2['composition'])

        letters = ['A', 'B', 'C', 'D']

        # ---------------------------------------------------------
        # TASK 2: THE SEVERITY TEST (MCQ)
        # ---------------------------------------------------------
        all_severities = ["MAJOR", "MODERATE", "MINOR", "NO INTERACTION"] 
        
        correct_severity = severity.upper()
        
        # Garantees the severity is correct 
        if correct_severity not in all_severities:
            all_severities.append(correct_severity)

        # Filter the wrong options
        wrong_severities = [s for s in all_severities if s != correct_severity]
        
        # Select until 3 wrongs to make A, B, C, D
        wrong_diag_options = random.sample(wrong_severities, min(3, len(wrong_severities)))

        diag_options = [correct_severity] + wrong_diag_options
        random.shuffle(diag_options)
        
        correct_letter_diag = letters[diag_options.index(correct_severity)]

        diag_options_text = ""
        for idx, opt in enumerate(diag_options):
            diag_options_text += f"{letters[idx]}. {opt}\n"

        input_text_2 = (
            f"###Question: A patient is taking {drug_curr_clean_2} [{current_med_2['medicine_name']}] for {cond_curr_2}. "
            f"I'm planning to prescribe {drug_prop_clean_2} [{proposed_med_2['medicine_name']}] for {cond_prop_2}. "
            f"Which of the following is the severity level associated with this drug combination?\n"
            f"###Options:\n{diag_options_text.strip()}"
        )

        test_dataset_mcq.append({
            "instruction": INSTRUCTION_MCQ,
            "input": input_text_2,
            "output": correct_letter_diag,
            "topic": "Severity_Classification"
        })

        # ---------------------------------------------------------
        # TASK 3: THE SUBSTITUTION TEST (MCQ)
        # ---------------------------------------------------------
        target_cond_2 = smart_lower(sols_2.get("condition_treated", ""))
        
        if sols_2.get("alternative_meds"):
            correct_med = sols_2["alternative_meds"][0]
            clean_correct_comp = strip_dosage(correct_med["composition"])
            
            wrong_options = []
            random.shuffle(all_meds_list) 

            for dist_med in all_meds_list:
                # List of uses and composition to compare
                dist_uses_list = [smart_lower(u) for u in dist_med.get('Uses', [])]
                dist_comp = strip_dosage(get_composition_str(dist_med))
                
                has_semantic_overlap = False
                for use in dist_uses_list:
                    # If the target disease is included in the use, OR the use is included in the target disease
                    if target_cond_2 in use or use in target_cond_2:
                        has_semantic_overlap = True
                        break # reject the medicine
                

                if not has_semantic_overlap:
                    if dist_comp != clean_correct_comp:
                        wrong_options.append({"medicine_name": dist_med["Medicine Name"], "composition": dist_comp})
                        
                if len(wrong_options) == 3:
                    break

            mcq_options = [{"medicine_name": correct_med["medicine_name"], "composition": clean_correct_comp}] + wrong_options
            random.shuffle(mcq_options)
            
            subs_options_text = ""
            correct_letter_subs = ""
            
            for idx, opt in enumerate(mcq_options):
                subs_options_text += f"{letters[idx]}. {opt['medicine_name']} ({opt['composition']})\n"
                if opt['medicine_name'] == correct_med["medicine_name"]:
                    correct_letter_subs = letters[idx]

            input_text_3 = (
                f"###Question: Patient taking {drug_curr_clean_2} [{current_med_2['medicine_name']}] requires medication for {target_cond_2}. "
                f"{drug_prop_clean_2} [{proposed_med_2['medicine_name']}] is contraindicated due to a {severity.lower()} drug-drug interaction. "
                f"Which of the following is the safest alternative to substitute the contraindicated medication?\n"
                f"###Options:\n{subs_options_text.strip()}"
            )

            test_dataset_mcq.append({
                "instruction": INSTRUCTION_MCQ,
                "input": input_text_3,
                "output": correct_letter_subs,
                "topic": "Safe_Substitution_Medicine"
            })

    out_open = "complete_cases_TEST_open.json"
    out_mcq = "complete_cases_TEST_mcq.json"
    
    with open(os.path.join(OUTPUT_DIR, out_open), 'w', encoding='utf-8') as f:
        json.dump(test_dataset_open, f, indent=4, ensure_ascii=False)
        
    with open(os.path.join(OUTPUT_DIR, out_mcq), 'w', encoding='utf-8') as f:
        json.dump(test_dataset_mcq, f, indent=4, ensure_ascii=False)

    print(f" -> Success! Generated {len(test_dataset_open)} Open QA pairs and {len(test_dataset_mcq)} MCQ pairs.")
    print(f" -> Quality Stats (Minimum score per pair): {stats_gold} Ideal (Tab+Tab) | {stats_silver} Good (Systemic) | {stats_bronze} Fallback (Topical)")
    
    if skipped_count > 0:
        print(f" -> Skipped {skipped_count} interaction groups (insufficient valid cases).")
    print(f" -> Saved standard JSON to: {OUTPUT_DIR}\n")
    
    #convert_json_to_jsonl(OUTPUT_DIR, [out_open, out_mcq])

if __name__ == "__main__":
    generate_test_qa_dataset()

--- [START] Generating Evaluation Dataset (Complete Cases TEST) ---
 -> Success! Generated 2 Open QA pairs and 4 MCQ pairs.
 -> Quality Stats (Minimum score per pair): 2 Ideal (Tab+Tab) | 0 Good (Systemic) | 0 Fallback (Topical)
 -> Saved standard JSON to: data_finetuning



### **- Clinical Management:**

**2 drugs to prescribe and the recommendation is just to monitor or adjust doses for example, doesn't include substitution**

#### **1. TRAIN and DEV datasets**

##### **1.1. JSON with all possible management cases**
This script processes the DDI leftovers interactions from Step 1 to identify and generate clinical scenarios requiring management strategies (e.g., dose adjustment, monitoring) rather than medication substitution.

**-Filtering:** Identifies management cases using a predefined phrase list (e.g., "monitor or adjust doses") from entries with "replacement_target": "BOTH" or "replacement_target": "N", where none of the drugs will be replaced.

In [9]:
import json
import os
import random
import re
import itertools

# Input: Reads leftovers from Step 1 (Complete Cases)
INPUT_DIR = os.path.join("data_generation_output", "complete_cases")

# Input: Medicines Data
MEDS_INPUT_DIR = os.path.join("..", "Medicines", "datasets_split")

# Output: Saves to a new folder for Step 2
OUTPUT_DIR = os.path.join("data_generation_output", "management_cases")

SEED = 42

# --- MANAGEMENT PHRASES LIST (STRICT MATCHING) ---
MANAGEMENT_PHRASES = {
    "monitor or adjust doses",
    "monitor or continue with caution", #"N"
    "use lowest possible sildenafil dose (25mg)",
    "start with lower doses",
    "none needed",  #"N"
    "space doses if needed",
    "use lowest effective dose",
    "space doses appropriately",
    "use one agent at a time",
    "monitor hr/bp; reduce dose",
    "use one anticoagulant only",
    "adjust doses carefully",
    "space doses by 4+ hours",
    "space doses by 5+ hours",
    "monitor or supplement potassium",
    "avoid combination if possible",  #"N"
    "space doses by 2+ hours",
    "withhold metformin before/after procedure",
    "use lower doses; monitor closely",
    "slow lamotrigine titration"
}


# --- HELPER FUNCTIONS ---

def clean_composition_text(text):
    text = re.sub(r'\s*\(.*?\)', '', text)
    return text.strip().lower()

def get_composition_str(med_dict):
    comp_list = med_dict.get('Composition', [])
    return ", ".join(comp_list) if comp_list else "Unknown"

def build_medicine_index(meds_data):
    index = {}
    meds_list = meds_data.values() if isinstance(meds_data, dict) else meds_data
    for med in meds_list:
        comp_list = med.get('Composition', [])
        if len(comp_list) != 1: continue 
        clean_key = clean_composition_text(comp_list[0])
        if clean_key not in index: index[clean_key] = []
        index[clean_key].append(med)
    return index

def get_unique_uses(source_med, comparison_med):
    """
    [HELPER] Clinical Logic (Anti-Redundancy)
    Returns a list of uses from 'source_med' that are NOT covered by 'comparison_med'.
    """
    unique_uses = []
    source_uses = source_med.get('Uses', [])
    # Create a string of all comparison uses for loose matching
    comp_uses_str = " ".join(comparison_med.get('Uses', [])).lower()
    
    for use in source_uses:
        # Check if this specific use text appears in the other med's uses
        if use.lower() not in comp_uses_str:
            unique_uses.append(use)
    return unique_uses

# --- GENERATION FUNCTION ---

def generate_management_dataset(leftover_file_in, meds_file, out_qa, out_left_phase2, out_unused, level_name):
    random.seed(SEED)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    leftover_path = os.path.join(INPUT_DIR, leftover_file_in)
    meds_path = os.path.join(MEDS_INPUT_DIR, meds_file)
    
    print(f"\n Generating {level_name} (Strict Management Cases)...")

    if not os.path.exists(leftover_path):
        print(f" Skipping {level_name}: Input file not found at {leftover_path}")
        return

    with open(leftover_path, 'r', encoding='utf-8') as f: leftovers = json.load(f)
    with open(meds_path, 'r', encoding='utf-8') as f: meds_data = json.load(f)
    
    med_index = build_medicine_index(meds_data)

    grouped_output = []
    phase2_leftovers = [] 
    used_med_names = set()
    
    total_cases_count = 0

    for item in leftovers:
        alt_text = item.get("Safer_alternative", "").strip()
        
        if alt_text.lower() not in MANAGEMENT_PHRASES: #exact correspondency
            phase2_leftovers.append(item)
            continue 

        d1_raw = item.get('drug_a', '').strip()
        d2_raw = item.get('drug_b', '').strip()
        d1_key, d2_key = d1_raw.lower(), d2_raw.lower()
        
        meds_a = med_index.get(d1_key)
        meds_b = med_index.get(d2_key)

        if not meds_a or not meds_b:
            phase2_leftovers.append(item)
            continue 

        interaction_group = {
            "interaction_pair": [d1_raw, d2_raw],
            "severity": item.get('severity_label'),
            "interaction_info": {
                "mechanism": item.get('mechanism'),
                "effect": item.get('effect'),
                "recommendation": alt_text,
                "rationale": item.get('rationale')
            },
            "generated_cases": [] 
        }

        # --- EXHAUSTIVE GENERATION ---
        ab_pairs = list(itertools.product(meds_a, meds_b))
        random.shuffle(ab_pairs) 

        for ma, mb in ab_pairs:
            # Basic validation
            if not ma.get('Uses') or not mb.get('Uses'): continue

            # --- STRICT LOGIC APPLIED HERE (Using get_unique_uses) ---
            
            # Identify "Unique Uses" for A
            valid_uses_for_A = get_unique_uses(ma, mb)
            
            # Identify "Unique Uses" for B
            valid_uses_for_B = get_unique_uses(mb, ma)

            # Check: If either list is empty
            if not valid_uses_for_A or not valid_uses_for_B:
                continue

            # Pick Random Contexts
            condition_a = random.choice(valid_uses_for_A)
            condition_b = random.choice(valid_uses_for_B) 

            case_entry = {
                "case_id": f"MANAGE_{level_name}_{len(grouped_output)}_{len(interaction_group['generated_cases']) + 1}",
                "patient_context": {
                    "medication": ma['Medicine Name'],
                    "composition": get_composition_str(ma),
                    "condition": condition_a
                },
                "proposed_context": {
                    "medication": mb['Medicine Name'],
                    "composition": get_composition_str(mb),
                    "condition": condition_b
                },
                "solution": {
                    "action": alt_text # Exact phrase match
                }
            }
            
            interaction_group["generated_cases"].append(case_entry)
            
            used_med_names.add(ma['Medicine Name'])
            used_med_names.add(mb['Medicine Name'])
            total_cases_count += 1
        
        # Save group if valid
        if interaction_group["generated_cases"]:
            grouped_output.append(interaction_group)
        else:
            phase2_leftovers.append(item)
    
    all_meds_list = meds_data.values() if isinstance(meds_data, dict) else meds_data
    unused_meds = [m for m in all_meds_list if m.get('Medicine Name') not in used_med_names]

    with open(os.path.join(OUTPUT_DIR, out_qa), 'w', encoding='utf-8') as f:
        json.dump(grouped_output, f, indent=4, ensure_ascii=False)
    
    with open(os.path.join(OUTPUT_DIR, out_left_phase2), 'w', encoding='utf-8') as f:
        json.dump(phase2_leftovers, f, indent=4, ensure_ascii=False)
        
    with open(os.path.join(OUTPUT_DIR, out_unused), 'w', encoding='utf-8') as f:
        json.dump(unused_meds, f, indent=4, ensure_ascii=False)

    print(f" {level_name}: Generated {len(grouped_output)} management groups.")
    print(f"   Total Scenarios: {total_cases_count}")
    print(f"   Saved to: {OUTPUT_DIR}")

if __name__ == "__main__":
    generate_management_dataset(
        leftover_file_in="DDI_Leftovers_TRAIN.json", 
        meds_file="dataset_meds_TRAIN_v2.json", 
        out_qa="management_cases_TRAIN.json", 
        out_left_phase2="DDI_Leftovers_Phase2_TRAIN.json",
        out_unused="unused_medicines_Phase2_TRAIN.json",
        level_name="TRAIN"
    )

    generate_management_dataset(
        leftover_file_in="DDI_Leftovers_DEV.json", 
        meds_file="dataset_meds_DEV_v2.json", 
        out_qa="management_cases_DEV.json", 
        out_left_phase2="DDI_Leftovers_Phase2_DEV.json",
        out_unused="unused_medicines_Phase2_DEV.json",
        level_name="DEV"
    )


 Generating TRAIN (Strict Management Cases)...
 TRAIN: Generated 16 management groups.
   Total Scenarios: 688
   Saved to: data_generation_output\management_cases

 Generating DEV (Strict Management Cases)...
 DEV: Generated 0 management groups.
   Total Scenarios: 0
   Saved to: data_generation_output\management_cases


##### **1.2 Formatting the QA Dataset**

Converts the management cases into the format (`instruction`, `input`, `output`) for fine-tuning

- **Scenario Selection:** The system evaluates all generated pairs and assigns weighted scores based on pharmaceutical formulations. The single highest-scoring case is extracted.
- **Formatting:** Detects when the source database misclassifies a management action (e.g., "Monitor blood pressure" or "Space doses") as a physiological effect. Instead of omitting the data or using unnatural bridge phrases, the script fuse the specific action with the general recommendation (e.g., merging "Monitor blood pressure" and "Monitor or adjust doses" into a single directive: "Monitor blood pressure or adjust doses.").

In [10]:
import json
import random
import re
import os


random.seed(42)

INSTRUCTIONS = [
    "You are a specialized AI assistant for healthcare. Supply reliable, evidence-based support directly to the physician. Review the proposed medication regimen for potential drug-drug interactions.",
    "Provide precise, actionable medical insights to the attending physician. Evaluate the clinical safety of adding this new medication to the patient's current therapy.",
    "You are an advanced AI medical assistant. Offer clear, professional advice to the treating physician. Provide clinical decision support regarding the following drug combination."
]

INPUT_TEMPLATES = [
    "Current medication: {med_a_comp_clean} [{med_a_brand}] for {cond_a}.\nProposed addition: {med_b_comp_clean} [{med_b_brand}] for {cond_b}.",
    "Patient is taking {med_a_comp_clean} [{med_a_brand}] for {cond_a}. I'm considering prescribing {med_b_comp_clean} [{med_b_brand}] for {cond_b}."
]

DISCLAIMER = "Clinical note: The significance of drug–drug interactions may vary depending on individual patient factors and specific drug characteristics, including dosage. A thorough clinical evaluation is recommended before any therapeutic adjustments."

# --- REGEX PATTERNS FOR SCORING ---
# Gold Level: Maximum Priority
SYSTEMIC_PATTERN = re.compile(r'\b(tablet|capsule)\b', re.IGNORECASE)
#Avoid if possible
TOPICAL_PATTERN = re.compile(r'\b(cream|gel|ointment|spray|drops?|lotion|shampoo|topical|soap|syrup|suppository)\b', re.IGNORECASE)

# --- FORMATTING HELPERS ---

def clean_clinical_text(text):
    if not text:
        return ""
    text = text.strip()
    
    text = text[0].upper() + text[1:]
    if not text.endswith('.'):
        text += '.'
        
    return text

def strip_dosage(composition_text):
    return re.sub(r'\s*\(.*?\)', '', composition_text).strip()

def smart_lower(text):
    if not text: 
        return ""
    parts = re.split(r'(\(.*?\))', str(text))
    result = ""
    for part in parts:
        if part.startswith('(') and part.endswith(')'):
            result += part 
        else:
            result += part.lower()  
    return result.strip()

def merge_action_effect(effect_text, action_text):
    # Helper function to merge redundant clinical directives
    if not effect_text:
        return action_text
        
    # Clean trailing periods to prevent double punctuation (..)
    e_clean = effect_text.strip().rstrip('.')
    a_clean = action_text.strip().rstrip('.')
    
    e_lower = e_clean.lower()
    a_lower = a_clean.lower()
    
    # CASE 1: Both start with "Monitor"
    if e_lower.startswith("monitor") and a_lower.startswith("monitor"):
        # e.g., effect: "Monitor blood pressure" | action: "Monitor or adjust doses"
        # Extract what comes after "Monitor " in the action string
        a_remainder = a_clean[len("Monitor"):].strip() 
        if a_remainder.startswith("or ") or a_remainder.startswith("and "):
            return f"{e_clean} {a_remainder}."
        else:
            return f"{e_clean}. {a_clean}."
            
    # CASE 2: Both involve "Space"
    elif "space" in e_lower and "space" in a_lower:
        # e.g., effect: "Space doses by 2+ hours" | action: "Space doses appropriately"
        # Extract the timeframe from the effect using regex
        time_match = re.search(r'\b(by\s+.*?)$', e_clean, re.IGNORECASE)
        if time_match:
            time_part = time_match.group(1)
            return f"Space doses appropriately ({time_part})."
        else:
            # Fallback if no timeframe is found
            return f"{a_clean} ({e_clean.lower()})."
            
    # CASE 3: Different verbs (e.g., "Monitor..." and "Start with...")
    else:
        # Combine sequentially, ensuring exactly one period per sentence
        return f"{e_clean}. {a_clean}."
    
# --- MAIN FUNCTION ---

def generate_management_alpaca_dataset(input_file, output_file, dataset_name):
    print(f"--- [START] Generating Management Alpaca Dataset for {dataset_name} ---")
    
    if not os.path.exists(input_file):
        print(f"   [SKIP] File '{input_file}' not found. Skipping {dataset_name}.")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        grouped_data = json.load(f)

    if not grouped_data:
        print(f"   [SKIP] File '{input_file}' is empty. Skipping {dataset_name}.")
        return

    alpaca_dataset = []
    
    stats_gold = 0
    stats_silver = 0
    stats_bronze = 0

    for group in grouped_data:
        cases = group.get("generated_cases")
        if not cases:
            continue

        severity = group.get("severity", "Unknown").upper()
        info = group["interaction_info"]
        
        mechanism = clean_clinical_text(info["mechanism"])
        effect_raw = clean_clinical_text(info["effect"])
        rationale = clean_clinical_text(info["rationale"])
        
        # --- SCORING SYSTEM FOR SELECTION ---
        def score_medication(med_name):
            if TOPICAL_PATTERN.search(med_name):
                return -10 # Heavy penalty for topical/syrup/injection
            elif SYSTEMIC_PATTERN.search(med_name):
                return 2   # Maximum bonus for Tablet/Capsule
            else:
                return 1   # Generic systemic fallback

        def score_case(c):
            a_name = c["patient_context"]["medication"]
            b_name = c["proposed_context"]["medication"]
            return score_medication(a_name) + score_medication(b_name)

        # Sort all generated cases from highest score to lowest
        cases.sort(key=score_case, reverse=True)
        
        # The best case will always be at index 0
        case = cases[0]
        
        # Update statistics based on the winning score
        best_score = score_case(case)
        if best_score == 4:
            stats_gold += 1   # Ideal: Tab + Tab
        elif best_score >= 2:
            stats_silver += 1 # Good: Systemic
        else:
            stats_bronze += 1 # Fallback: At least one Topical involved
        # --------------------------------------------------------------

        med_A = case["patient_context"]
        med_B = case["proposed_context"]
        action_text = case["solution"]["action"]

        drug_a_clean = strip_dosage(med_A['composition'])
        drug_b_clean = strip_dosage(med_B['composition'])

        instruction = random.choice(INSTRUCTIONS)
        input_template = random.choice(INPUT_TEMPLATES)
        
        # Safely extract conditions
        cond_a_lower = smart_lower(str(med_A.get('condition', '')))
        cond_b_lower = smart_lower(str(med_B.get('condition', '')))
        
        input_text = input_template.format(
            med_a_brand=med_A['medication'], med_a_comp_clean=drug_a_clean, cond_a=cond_a_lower,
            med_b_brand=med_B['medication'], med_b_comp_clean=drug_b_clean, cond_b=cond_b_lower
        )

        action_formatted = clean_clinical_text(action_text)
        action_lower = action_formatted.lower()
        
        if "none needed" in action_lower:
            output_text = f"Co-administration of {drug_a_clean} and {drug_b_clean} is considered generally safe. No significant interaction expected."
        else:
            # Universal Neutral Header
            header = f"Clinical attention is required for the co-administration of {drug_a_clean} and {drug_b_clean}."
            
            output_lines = [
                header + "\n",
                f"Interaction details between {drug_a_clean} and {drug_b_clean}:",
                f"**Severity:** {severity}",
                f"**Mechanism:** {mechanism}"
            ]

            is_action_effect = effect_raw.lower().startswith(("monitor", "space"))
            
            if is_action_effect:
                merged_recommendation = merge_action_effect(effect_raw, action_formatted)
                output_lines.append(f"**Recommendations:** {merged_recommendation}")
            else:
                if effect_raw:
                    output_lines.append(f"**Effect:** {effect_raw}")
                output_lines.append(f"**Recommendations:** {action_formatted}")


            output_lines.append(f"**Rationale:** {rationale}")
            output_lines.append(f"\n*{DISCLAIMER}*")
            output_text = "\n".join(output_lines)

        alpaca_dataset.append({
            "instruction": instruction,
            "input": input_text,
            "output": output_text
        })

    with open(output_file, 'w', encoding='utf-8') as f: 
        json.dump(alpaca_dataset, f, indent=4, ensure_ascii=False)

    print(f"   Success! Generated {len(alpaca_dataset)} QA pairs.")
    print(f"   -> Quality Stats: {stats_gold} Ideal (Tab+Tab) | {stats_silver} Good (Systemic) | {stats_bronze} Fallback (Topical)")
    print(f"   -> Saved standard JSON to: {output_file}\n")


if __name__ == "__main__":
    TARGET_DIR = "data_finetuning"
    os.makedirs(TARGET_DIR, exist_ok=True)

    generate_management_alpaca_dataset(
        input_file="data_generation_output/management_cases/management_cases_TRAIN.json",
        output_file=os.path.join(TARGET_DIR, "management_cases_TRAIN.json"),
        dataset_name="TRAIN"
    )
    
    generate_management_alpaca_dataset(
        input_file="data_generation_output/management_cases/management_cases_DEV.json",
        output_file=os.path.join(TARGET_DIR, "management_cases_DEV.json"),
        dataset_name="DEV"
    )

    files_to_convert = [
        "management_cases_TRAIN.json",
        "management_cases_DEV.json"
    ]
    #convert_json_to_jsonl(TARGET_DIR, files_to_convert)

--- [START] Generating Management Alpaca Dataset for TRAIN ---
   Success! Generated 16 QA pairs.
   -> Quality Stats: 12 Ideal (Tab+Tab) | 3 Good (Systemic) | 1 Fallback (Topical)
   -> Saved standard JSON to: data_finetuning\management_cases_TRAIN.json

--- [START] Generating Management Alpaca Dataset for DEV ---
   [SKIP] File 'data_generation_output/management_cases/management_cases_DEV.json' is empty. Skipping DEV.


#### **2. TEST dataset**

This script processes leftovers from the first step and generate an evaluation dataset. 
The generated QA pairs are partitioned into two files:
  * **Open-Ended QA (`_open.json`)**
  * **Multiple Choice Questions (`_mcq.json`)**
  
**Key Logic**
* **Scenario Selection:** Uses a scoring algorithm to prioritize systemic interactions. Solid oral dose forms (tablets/capsules) receive maximum priority (+2 points), while localized forms (e.g., creams, gels) are penalized (-10 points). The top 2 highest-scoring distinct cases are extracted for testing.
* **Formatting:** Detects when a database misclassifies an action (e.g., "Monitor blood pressure") as a effect. In this case it fuse the specific action with the general recommendation into a single directive.
* **MCQ Evaluation (Case 2):**
  * *Task 2 (Severity Classification)*
  * *Task 3 (Clinical Action)* 

In [11]:
import json
import os
import random
import re
import itertools

INPUT_DIR = os.path.join("data_generation_output", "complete_cases")
MEDS_INPUT_DIR = os.path.join("..", "Medicines", "datasets_split")

RAW_OUTPUT_DIR = os.path.join("data_generation_output", "management_cases")
FINETUNING_DIR = "data_finetuning"

SEED = 42

# Only these exact phrases occur in the Test Set leftovers
MANAGEMENT_PHRASES_TEST = {
    "space doses appropriately",
    "monitor or continue with caution",
    "none needed",
    "monitor or adjust doses"
}

# --- REGEX PATTERNS FOR SCORING ---
# Gold Level: Maximum Priority
SYSTEMIC_PATTERN = re.compile(r'\b(tablet|capsule)\b', re.IGNORECASE)
# Avoid if possible
TOPICAL_PATTERN = re.compile(r'\b(cream|gel|ointment|spray|drops?|lotion|shampoo|topical|soap|syrup|suppository)\b', re.IGNORECASE)

INSTRUCTIONS_OPEN_QA = [
    "You are a specialized AI assistant for healthcare. Supply reliable, evidence-based support directly to the physician. Review the proposed medication regimen for potential drug-drug interactions.",
    "Provide precise, actionable medical insights to the attending physician. Evaluate the clinical safety of adding this new medication to the patient's current therapy.",
    "You are an advanced AI medical assistant. Offer clear, professional advice to the treating physician. Provide clinical decision support regarding the following drug combination."
]

INPUT_TEMPLATES_OPEN_QA = [
    "Current medication: {med_a_comp} [{med_a_brand}] for {cond_a}.\nProposed addition: {med_b_comp} [{med_b_brand}] for {cond_b}.",
    "Patient is taking {med_a_comp} [{med_a_brand}] for {cond_a}. I'm considering prescribing {med_b_comp} [{med_b_brand}] for {cond_b}."
]

DISCLAIMER = "Clinical note: The significance of drug–drug interactions may vary depending on individual patient factors and specific drug characteristics, including dosage. A thorough clinical evaluation is recommended before any therapeutic adjustments."

# Task 2 & 3: MCQ Instruction
INSTRUCTION_MCQ = "Read the question and options carefully. Output ONLY the letter (A, B, C, or D) corresponding to the correct answer."

# --- FORMATTING HELPERS ---

def clean_clinical_text(text):
    if not text:
        return ""
    text = text.strip()
    
    text = text[0].upper() + text[1:]
    if not text.endswith('.'):
        text += '.'
        
    return text

def clean_composition_text(text):
    text = re.sub(r'\s*\(.*?\)', '', text)
    return text.strip().lower()

def strip_dosage(composition_text):
    return re.sub(r'\s*\(.*?\)', '', composition_text).strip()

def get_composition_str(med_dict):
    comp_list = med_dict.get('Composition', [])
    return ", ".join(comp_list) if comp_list else "Unknown"

def smart_lower(text):
    if not text: 
        return ""
    parts = re.split(r'(\(.*?\))', str(text))
    result = ""
    for part in parts:
        if part.startswith('(') and part.endswith(')'):
            result += part 
        else:
            result += part.lower()  
    return result.strip()

def merge_action_effect(effect_text, action_text):
    # Helper function to merge redundant clinical directives
    if not effect_text:
        return action_text
        
    e_clean = effect_text.strip().rstrip('.')
    a_clean = action_text.strip().rstrip('.')
    
    e_lower = e_clean.lower()
    a_lower = a_clean.lower()
    
    if e_lower.startswith("monitor") and a_lower.startswith("monitor"):
        a_remainder = a_clean[len("Monitor"):].strip() 
        if a_remainder.startswith("or ") or a_remainder.startswith("and "):
            return f"{e_clean} {a_remainder}."
        else:
            return f"{e_clean}. {a_clean}."
            
    elif "space" in e_lower and "space" in a_lower:
        # Extract the timeframe from the effect using regex (e.g., "by 2+ hours")
        time_match = re.search(r'\b(by\s+.*?)$', e_clean, re.IGNORECASE)
        if time_match:
            time_part = time_match.group(1)
            return f"Space doses appropriately ({time_part})."
        else:
            # If there's no "by", extract whatever comes after "Space doses" in the effect
            e_remainder = re.sub(r'(?i)^space\s+(doses\s+)?', '', e_clean).strip()
            
            if e_remainder:
                # e.g., e_remainder = "or use liquid formulation"
                if e_remainder.startswith("or ") or e_remainder.startswith("and "):
                    return f"Space doses appropriately {e_remainder}."
                else:
                    # Fallback just in case it doesn't have a conjunction
                    return f"Space doses appropriately and {e_remainder}."
            else:
                # If the effect was literally just "Space doses", avoid repetition
                return f"Space doses appropriately."
            
    else:
        return f"{e_clean}. {a_clean}."

# --- INDEXING & FILTERING ---

def build_medicine_index(meds_data):
    index = {}
    meds_list = meds_data.values() if isinstance(meds_data, dict) else meds_data
    for med in meds_list:
        comp_list = med.get('Composition', [])
        if len(comp_list) != 1: continue 
        clean_key = clean_composition_text(comp_list[0])
        if clean_key not in index: index[clean_key] = []
        index[clean_key].append(med)
    return index

def get_unique_uses(source_med, comparison_med):
    unique_uses = []
    source_uses = source_med.get('Uses', [])
    comp_uses_str = " ".join(comparison_med.get('Uses', [])).lower()
    for use in source_uses:
        if use.lower() not in comp_uses_str:
            unique_uses.append(use)
    return unique_uses

# --- MAIN GENERATION FUNCTION ---

def generate_management_test_dataset(leftover_ddi_in, leftover_meds_in, meds_file, output_filename, leftover_ddi_out, leftover_meds_out):
    random.seed(SEED)
    os.makedirs(RAW_OUTPUT_DIR, exist_ok=True)
    os.makedirs(FINETUNING_DIR, exist_ok=True)
    
    leftover_path = os.path.join(INPUT_DIR, leftover_ddi_in)
    unused_meds_path = os.path.join(INPUT_DIR, leftover_meds_in)
    meds_path = os.path.join(MEDS_INPUT_DIR, meds_file)
    
    print(f"\n--- [START] Generating TEST Management Dataset ---")

    if not os.path.exists(leftover_path) or not os.path.exists(meds_path):
        print(" [ERROR] Required files not found. Check paths.")
        return

    with open(leftover_path, 'r', encoding='utf-8') as f: leftovers = json.load(f)
    with open(meds_path, 'r', encoding='utf-8') as f: meds_data = json.load(f)
    
    unused_meds_data = {}
    if os.path.exists(unused_meds_path):
        with open(unused_meds_path, 'r', encoding='utf-8') as f: 
            unused_meds_data = json.load(f)
    
    med_index = build_medicine_index(meds_data)
    
    # ---------------------------------------------------------
    # STEP 1: GENERATE ALL CASES AND SAVE RAW DATA & LEFTOVERS
    # ---------------------------------------------------------
    grouped_output = []
    phase2_leftovers = []
    used_med_names = set()

    # Create global pools for MCQs (Tasks 2 and 3)
    global_diagnosis_pool = []
    global_rec_pool = []

    for item in leftovers:
        alt_text = item.get("Safer_alternative", "").strip()
        
        if alt_text.lower() not in MANAGEMENT_PHRASES_TEST:
            phase2_leftovers.append(item)
            continue 

        # Build Recommendation Pool
        effect_raw_pool = clean_clinical_text(item.get('effect'))
        action_formatted_pool = clean_clinical_text(alt_text)
        is_action_effect_pool = effect_raw_pool.lower().startswith(("monitor", "space"))
        
        if is_action_effect_pool:
            merged_rec = merge_action_effect(effect_raw_pool, action_formatted_pool)
            if merged_rec not in global_rec_pool:
                global_rec_pool.append(merged_rec)
        else:
            if action_formatted_pool not in global_rec_pool:
                global_rec_pool.append(action_formatted_pool)


        d1_raw = item.get('drug_a', '').strip()
        d2_raw = item.get('drug_b', '').strip()
        
        meds_a = med_index.get(d1_raw.lower())
        meds_b = med_index.get(d2_raw.lower())

        if not meds_a or not meds_b:
            phase2_leftovers.append(item)
            continue 

        valid_scenarios = []
        ab_pairs = list(itertools.product(meds_a, meds_b))
        random.shuffle(ab_pairs) 

        for ma, mb in ab_pairs:
            if not ma.get('Uses') or not mb.get('Uses'): continue
            
            valid_uses_A = get_unique_uses(ma, mb)
            valid_uses_B = get_unique_uses(mb, ma)

            if not valid_uses_A or not valid_uses_B: continue

            valid_scenarios.append({
                "patient_context": {
                    "medication": ma['Medicine Name'],
                    "composition": get_composition_str(ma),
                    "condition": random.choice(valid_uses_A)
                },
                "proposed_context": {
                    "medication": mb['Medicine Name'],
                    "composition": get_composition_str(mb),
                    "condition": random.choice(valid_uses_B) 
                },
                "solution": {
                    "action": alt_text
                }
            })

        if not valid_scenarios:
            phase2_leftovers.append(item)
            continue
            
        for scenario in valid_scenarios:
            used_med_names.add(scenario["patient_context"]["medication"])
            used_med_names.add(scenario["proposed_context"]["medication"])
            
        grouped_output.append({
            "interaction_pair": [d1_raw, d2_raw],
            "severity": item.get('severity_label', 'Unknown').upper(),
            "interaction_info": {
                "mechanism": item.get('mechanism'),
                "effect": item.get('effect'),
                "recommendation": alt_text,
                "rationale": item.get('rationale')
            },
            "generated_cases": valid_scenarios
        })

    raw_out_path = os.path.join(RAW_OUTPUT_DIR, output_filename)
    with open(raw_out_path, 'w', encoding='utf-8') as f:
        json.dump(grouped_output, f, indent=4, ensure_ascii=False)

    out_ddi_left_path = os.path.join(RAW_OUTPUT_DIR, leftover_ddi_out)
    with open(out_ddi_left_path, 'w', encoding='utf-8') as f:
        json.dump(phase2_leftovers, f, indent=4, ensure_ascii=False)

    if unused_meds_data:
        if isinstance(unused_meds_data, dict):
            phase2_unused_meds = {k: v for k, v in unused_meds_data.items() if v.get("Medicine Name") not in used_med_names}
        else:
            phase2_unused_meds = [m for m in unused_meds_data if m.get("Medicine Name") not in used_med_names]
            
        out_unused_meds_path = os.path.join(RAW_OUTPUT_DIR, leftover_meds_out)
        with open(out_unused_meds_path, 'w', encoding='utf-8') as f:
            json.dump(phase2_unused_meds, f, indent=4, ensure_ascii=False)

    # ---------------------------------------------------------
    # STEP 2: BUILD QA FROM RAW DATA
    # ---------------------------------------------------------
    
    test_dataset_open = []
    test_dataset_mcq = []

    # Counters for Quality Stats
    stats_gold = 0
    stats_silver = 0
    stats_bronze = 0

    for group in grouped_output:
        severity = group["severity"]
        info = group["interaction_info"]
        
        mechanism = clean_clinical_text(info["mechanism"])
        effect_raw = clean_clinical_text(info["effect"])
        rationale = clean_clinical_text(info["rationale"])
        action_formatted = clean_clinical_text(info["recommendation"])

        cases = group["generated_cases"]
        
        # --- SCORING SYSTEM FOR SELECTION ---
        def score_medication(med_name):
            if TOPICAL_PATTERN.search(med_name):
                return -10 
            elif SYSTEMIC_PATTERN.search(med_name):
                return 2   
            else:
                return 1   

        def score_case(c):
            a_name = c["patient_context"]["medication"]
            b_name = c["proposed_context"]["medication"]
            return score_medication(a_name) + score_medication(b_name)

        cases.sort(key=score_case, reverse=True)
        
        case_1 = cases[0]
        case_2 = cases[1] if len(cases) > 1 else cases[0]
        
        weakest_score = min(score_case(case_1), score_case(case_2))
        
        if weakest_score == 4: stats_gold += 1   
        elif weakest_score >= 2: stats_silver += 1 
        else: stats_bronze += 1 

        # =========================================================
        # TASK 1: OPEN-ENDED QA (Using Case 1)
        # =========================================================
        med_A_1 = case_1["patient_context"]
        med_B_1 = case_1["proposed_context"]
        
        drug_a_clean_1 = strip_dosage(med_A_1["composition"])
        drug_b_clean_1 = strip_dosage(med_B_1["composition"])
        
        cond_a_lower_1 = smart_lower(str(med_A_1.get("condition", "")))
        cond_b_lower_1 = smart_lower(str(med_B_1.get("condition", "")))

        input_template = random.choice(INPUT_TEMPLATES_OPEN_QA)
        input_text_1 = input_template.format(
            med_a_comp=drug_a_clean_1, med_a_brand=med_A_1['medication'], cond_a=cond_a_lower_1,
            med_b_comp=drug_b_clean_1, med_b_brand=med_B_1['medication'], cond_b=cond_b_lower_1
        )

        if "none needed" in action_formatted.lower():
            output_text_1 = f"Co-administration of {drug_a_clean_1} and {drug_b_clean_1} is considered generally safe. No significant interaction expected."
        else:
            header = f"Clinical attention is required for the co-administration of {drug_a_clean_1} and {drug_b_clean_1}."
            output_lines_1 = [
                header + "\n",
                f"Interaction details between {drug_a_clean_1} and {drug_b_clean_1}:",
                f"- **Severity:** {severity}",
                f"- **Mechanism:** {mechanism}"
            ]
            
            is_action_effect = effect_raw.lower().startswith(("monitor", "space"))
            
            if not is_action_effect and effect_raw:
                output_lines_1.append(f"- **Effect:** {effect_raw}")
                output_lines_1.append(f"- **Recommendation:** {action_formatted}")
            elif is_action_effect:
                merged_recommendation = merge_action_effect(effect_raw, action_formatted)
                output_lines_1.append(f"- **Recommendation:** {merged_recommendation}")
            else:
                output_lines_1.append(f"- **Recommendation:** {action_formatted}")
                
            output_lines_1.append(f"- **Rationale:** {rationale}")
            output_lines_1.append(f"\n*{DISCLAIMER}*")
            output_text_1 = "\n".join(output_lines_1)

        test_dataset_open.append({
            "instruction": random.choice(INSTRUCTIONS_OPEN_QA),
            "input": input_text_1,
            "output": output_text_1
        })

        # =========================================================
        # Setup variables for Case 2 (Used for both Task 2 & 3)
        # =========================================================
        med_A_2 = case_2["patient_context"]
        med_B_2 = case_2["proposed_context"]
        
        drug_a_clean_2 = strip_dosage(med_A_2["composition"])
        drug_b_clean_2 = strip_dosage(med_B_2["composition"])
        
        cond_a_lower_2 = smart_lower(str(med_A_2.get("condition", "")))
        cond_b_lower_2 = smart_lower(str(med_B_2.get("condition", "")))

        letters = ['A', 'B', 'C', 'D']

        # =========================================================
        # TASK 2: MedQA-Style MCQ (Severity Classification)
        # =========================================================
        all_severities = ["MAJOR", "MODERATE", "MINOR", "NO INTERACTION"] 
        
        correct_severity = severity.upper()
        if correct_severity not in all_severities:
            all_severities.append(correct_severity)

        wrong_severities = [s for s in all_severities if s != correct_severity]
        wrong_diag_options = random.sample(wrong_severities, min(3, len(wrong_severities)))

        diag_options = [correct_severity] + wrong_diag_options
        random.shuffle(diag_options)
        
        correct_letter_diag = letters[diag_options.index(correct_severity)]

        diag_options_text = ""
        for idx, opt in enumerate(diag_options):
            diag_options_text += f"{letters[idx]}. {opt}\n"

        input_text_2 = (
            f"###Question: A patient is taking {drug_a_clean_2} [{med_A_2['medication']}] for {cond_a_lower_2}. "
            f"I'm considering prescribing {drug_b_clean_2} [{med_B_2['medication']}] for {cond_b_lower_2}. "
            f"Which of the following is the severity level associated with this drug combination?\n"
            f"###Options:\n{diag_options_text.strip()}"
        )

        test_dataset_mcq.append({
            "instruction": INSTRUCTION_MCQ,
            "input": input_text_2,
            "output": correct_letter_diag,
            "topic": "Severity_Classification"
        })

        # =========================================================
        # TASK 3: MCQ (Management Action)
        # =========================================================
        
        # Establish the correct option dynamically (merging if needed)
        is_action_effect_mcq = effect_raw.lower().startswith(("monitor", "space"))
        correct_rec = merge_action_effect(effect_raw, action_formatted) if is_action_effect_mcq else action_formatted

        wrong_rec_options = []
        random.shuffle(global_rec_pool)
        for rec in global_rec_pool:
            if rec != correct_rec and rec not in wrong_rec_options:
                wrong_rec_options.append(rec)
            if len(wrong_rec_options) == 3:
                break

        rec_options = [correct_rec] + wrong_rec_options
        random.shuffle(rec_options)

        options_text_rec = ""
        correct_letter_rec = ""

        for idx, opt in enumerate(rec_options):
            options_text_rec += f"{letters[idx]}. {opt}\n"
            if opt == correct_rec:
                correct_letter_rec = letters[idx]

        input_text_3 = (
            f"###Question: A patient is taking {drug_a_clean_2} [{med_A_2['medication']}] for {cond_a_lower_2}. "
            f"I'm considering prescribing {drug_b_clean_2} [{med_B_2['medication']}] for {cond_b_lower_2}. "
            f"Which of the following is the most appropriate clinical management?\n"
            f"###Options:\n{options_text_rec.strip()}"
        )

        test_dataset_mcq.append({
            "instruction": INSTRUCTION_MCQ,
            "input": input_text_3,
            "output": correct_letter_rec,
            "topic": "Clinical_Management"
        })

    open_filename = output_filename.replace(".json", "_open.json")
    open_out_path = os.path.join(FINETUNING_DIR, open_filename)
    with open(open_out_path, 'w', encoding='utf-8') as f:
        json.dump(test_dataset_open, f, indent=4, ensure_ascii=False)
        
    mcq_filename = output_filename.replace(".json", "_mcq.json")
    mcq_out_path = os.path.join(FINETUNING_DIR, mcq_filename)
    with open(mcq_out_path, 'w', encoding='utf-8') as f:
        json.dump(test_dataset_mcq, f, indent=4, ensure_ascii=False)

    print(f" -> Success! Generated {len(test_dataset_open)} Open QA pairs and {len(test_dataset_mcq)} MCQ pairs (Tasks 2 & 3).")
    print(f" -> Quality Stats (Minimum score per pair): {stats_gold} Ideal (Tab+Tab) | {stats_silver} Good (Systemic) | {stats_bronze} Fallback (Topical)")
    print(f" -> Saved format JSONs to: {FINETUNING_DIR}\n")


if __name__ == "__main__":
    generate_management_test_dataset(
        leftover_ddi_in="DDI_Leftovers_TEST.json", 
        leftover_meds_in="unused_medicines_TEST.json",
        meds_file="dataset_meds_TEST_v2.json", 
        output_filename="management_cases_TEST.json",
        leftover_ddi_out="DDI_leftovers_Phase2_TEST.json",
        leftover_meds_out="unused_medicines_Phase2_TEST.json"
    )


--- [START] Generating TEST Management Dataset ---
 -> Success! Generated 2 Open QA pairs and 4 MCQ pairs (Tasks 2 & 3).
 -> Quality Stats (Minimum score per pair): 1 Ideal (Tab+Tab) | 0 Good (Systemic) | 1 Fallback (Topical)
 -> Saved format JSONs to: data_finetuning



### **-Unmatched DDIs (Leftovers):**
**Pharmacological reasoning between DDis leftovers that couldn't be matched with commercial brands or specific clinical conditions**

#### **1. TRAIN and DEV datasets**

##### **1.1 QA generation**

This script processes the raw active substance rules that weren't used in previous generations. Since these cases lack commercial brands and clinical conditions, the focus is strictly on substance-to-substance interactions.

* **Clinical Logic:** If the dataset rule dictates replacing Drug A, the script dynamically assigns Drug B as the patient's current medication and Drug A as the "proposed addition." For symmetric targets (BOTH or N), it randomly assigns the roles.
* **Formatting:** 
    - Prevents fragmented phrasing when the database provides only a drug name for substitution. If the target involves replacing a drug and the recommendation lacks a verb, the script dynamically injects the prefix `"Consider substitute: "` (e.g., transforming `"Acetaminophen"` into `"Consider substitute: Acetaminophen"`).
    - If the script detects two completely distinct clinical actions (e.g., "Monitor blood pressure" and "Use one agent at a time"), it automatically separates them into a clean, bulleted list.
    - Detects and cleans shorthand database notations, such as a trailing `"; monitor"` in the `Effect` field, redistributing the monitoring action directly into the recommendation block as a formalized bullet point.
    - Injects a standardized Clinical Disclaimer exclusively for interactions requiring clinical attention.

In [12]:
import json
import random
import re
import os

random.seed(42)

INSTRUCTIONS = [
    "Offer clear, professional clinical support to the treating physician. Analyze the pharmacological interaction between the specified active substances.",
    "You are a specialized clinical AI. Provide precise, actionable medical insights to the attending physician. Evaluate the clinical safety of combining these two drugs.",
    "You are a specialized AI assistant for healthcare. Supply reliable, evidence-based support directly to the physician. Identify any drug-drug interactions for this combination and recommend clinical adjustments.",
    "Act as an expert clinical assistant. Deliver brief, evidence-based recommendations to the healthcare provider. Provide a pharmacological safety assessment for this drug pairing."
]

# For targets A or B (current vs Proposed)
INPUT_TEMPLATES_ASYMMETRIC = [
    "Is it safe to prescribe {proposed_drug} to a patient already taking {current_drug}?",
    "I'm considering adding {proposed_drug} to a regimen that includes {current_drug}. Are there any contraindications?",
    "Patient is currently on {current_drug}. Can I start them on {proposed_drug}?"
]

# For targets BOTH or N (General combination)
INPUT_TEMPLATES_SYMMETRIC = [
    "Evaluate the interaction risk of co-administering {current_drug} and {proposed_drug}.",
    "Review the clinical safety of combining {current_drug} and {proposed_drug}.",
    "Are there any pharmacological concerns with the concurrent use of {current_drug} and {proposed_drug}?"
]

DISCLAIMER = "Clinical note: The significance of drug–drug interactions may vary depending on individual patient factors and specific drug characteristics, including dosage. A thorough clinical evaluation is recommended before any therapeutic adjustments."

# --- FORMATTING HELPERS ---

def clean_clinical_text(text):
    if not text:
        return ""
    text = text.strip()
    
    text = text[0].upper() + text[1:]
    if not text.endswith('.'):
        text += '.'
        
    return text

def merge_action_effect(effect_text, action_text):
    # Helper function to merge redundant clinical directives
    if not effect_text:
        return action_text
        
    e_clean = effect_text.strip().rstrip('.')
    a_clean = action_text.strip().rstrip('.')
    
    e_lower = e_clean.lower()
    a_lower = a_clean.lower()
    
    # Create a clean 'core' of our own prefix so as not to break the redundancy check
    a_core = a_lower.replace("consider substitute: ", "").strip()
    e_core = e_lower.replace("consider substitute: ", "").strip()
    
    # CASE 1: Both start with "Monitor"
    if e_lower.startswith("monitor") and a_lower.startswith("monitor"):
        a_remainder = a_clean[len("Monitor"):].strip() 
        if a_remainder.startswith("or ") or a_remainder.startswith("and "):
            return f"{e_clean} {a_remainder}."
        else:
            # CREATE BULLET POINTS FOR INDEPENDENT SENTENCES
            return f"\n- {e_clean}.\n- {a_clean}."
            
    # CASE 2: Both involve "Space"
    elif "space" in e_lower and "space" in a_lower:
        time_match = re.search(r'\b(by\s+.*?)$', e_clean, re.IGNORECASE)
        if time_match:
            time_part = time_match.group(1)
            return f"Space doses appropriately ({time_part})."
        else:
            e_remainder = re.sub(r'(?i)^space\s+(doses\s+)?', '', e_clean).strip()
            if e_remainder:
                if e_remainder.startswith("or ") or e_remainder.startswith("and "):
                    return f"Space doses appropriately {e_remainder}."
                else:
                    return f"Space doses appropriately and {e_remainder}."
            else:
                return f"Space doses appropriately."
                
    # CASE 3: Complete redundancy ("Consider substitute: ...")
    elif a_core in e_core:
        return f"{e_clean}."
    elif e_core in a_core:
        return f"{a_clean}."
            
    # CASE 4: Different verbs (CREATE BULLET POINTS)
    else:
        return f"\n- {e_clean}.\n- {a_clean}."

# --- MAIN GENERATION FUNCTION ---

def generate_leftovers_alpaca_dataset(input_file, output_file, dataset_name):
    print(f"--- [START] Generating Realistic QA Dataset for {dataset_name} (Leftovers) ---")
    
    if not os.path.exists(input_file):
        print(f"   [SKIP] File '{input_file}' not found. Skipping {dataset_name}.")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if not data:
        print(f"   [SKIP] File '{input_file}' is empty. Skipping {dataset_name}.")
        return

    alpaca_dataset = []

    for item in data:
        drug_a = item.get("drug_a", "").strip()
        drug_b = item.get("drug_b", "").strip()
        
        # Safely extract severity using both possible key names
        severity = item.get("severity_label", item.get("severity", "Unknown")).upper()
        
        mechanism = clean_clinical_text(item.get("mechanism", ""))
        effect_raw = clean_clinical_text(item.get("effect", ""))
        rationale = clean_clinical_text(item.get("rationale", ""))
        alt_text = item.get("Safer_alternative", "").strip()
        target = item.get("replacement_target", "").upper()

        recommendation = clean_clinical_text(alt_text)

        # --- 1. CLINICAL LOGIC ---
        if target == "A":
            current_drug = drug_b
            proposed_drug = drug_a
        elif target == "B":
            current_drug = drug_a
            proposed_drug = drug_b
        else:
            # For "BOTH", "N", or empty targets
            if random.choice([True, False]):
                current_drug = drug_a
                proposed_drug = drug_b
            else:
                current_drug = drug_b
                proposed_drug = drug_a

        # --- 2. SELECT INSTRUCTION ---
        instruction = random.choice(INSTRUCTIONS)

        # --- 3. BUILD DYNAMIC INPUT ---
        if target in ["A", "B"]:
            input_template = random.choice(INPUT_TEMPLATES_ASYMMETRIC)
        else:
            input_template = random.choice(INPUT_TEMPLATES_SYMMETRIC)
            
        input_text = input_template.format(current_drug=current_drug, proposed_drug=proposed_drug)

        # --- 4. BUILD OUTPUT ---
        alt_text_lower = recommendation.lower()
        
        if "none needed" in alt_text_lower:
            output_text = f"Co-administration of {drug_a} and {drug_b} is considered generally safe. No significant interaction expected."
        else:
            rec_label = "**Recommendations**"

            # Universal Header for ALL interactions
            header = f"Clinical attention is required for the co-administration of {drug_a} and {drug_b}."
            
            output_lines = [
                header + "\n",
                f"Interaction details between {drug_a} and {drug_b}:",
                f"**Severity:** {severity}",
                f"**Mechanism:** {mechanism}"
            ]
            
            # --- MERGE LOGIC ---
            # --- FIX: CATCH TRAILING "; MONITOR" IN EFFECT ---
            if re.search(r';\s*monitor\.*$', effect_raw, re.IGNORECASE):
                # 1. Strip "; monitor" from the effect and re-clean it
                effect_raw = re.sub(r';\s*monitor\.*$', '', effect_raw, flags=re.IGNORECASE).strip()
                effect_raw = clean_clinical_text(effect_raw)
                
                # 2. Apply the "Consider substitute" heuristic since we are making bullets
                rec_lower = recommendation.lower()
                action_keywords = ("monitor", "space", "avoid", "use", "discontinue", "maintain", "adjust", "start", "consider", "other", "alternative", "loop", "withhold", "reduce")
                if target in ["A", "B"] and not rec_lower.startswith(action_keywords):
                    recommendation = f"Consider substitute: {recommendation}"
                
                # 3. Transform Recommendation into a bulleted list
                recommendation = f"\n- Monitor.\n- {recommendation}"
            # ---------------------------------------------------
            # Added "maintain" to catch cases like "Maintain consistent dietary intake"
            is_action_effect = effect_raw.lower().startswith(("monitor", "space", "maintain"))
            
            # Only inject "Consider substitute" if we are merging the sentences
            if is_action_effect and target in ["A", "B"]:
                rec_lower = recommendation.lower()
                action_keywords = ("monitor", "space", "avoid", "use", "discontinue", "maintain", "adjust", "start", "consider", "other", "alternative", "loop", "withhold", "reduce")
                if not rec_lower.startswith(action_keywords):
                    recommendation = f"Consider substitute: {recommendation}"

            if not is_action_effect and effect_raw:
                # Normal scenario: separate lines
                output_lines.append(f"**Effect:** {effect_raw}")
                # Check if recommendation was turned into a bulleted list above
                if recommendation.startswith("\n"):
                    output_lines.append(f"{rec_label}:{recommendation}")
                else:
                    output_lines.append(f"{rec_label}: {recommendation}")
            elif is_action_effect:
                # Merged scenario: Check if it returned a bulleted list or a single string
                merged_recommendation = merge_action_effect(effect_raw, recommendation)
                if merged_recommendation.startswith("\n"):
                    output_lines.append(f"{rec_label}:{merged_recommendation}")
                else:
                    output_lines.append(f"{rec_label}: {merged_recommendation}")
            else:
                output_lines.append(f"{rec_label}: {recommendation}")

            output_lines.append(f"**Rationale:** {rationale}")

            output_lines.append(f"\n*{DISCLAIMER}*")
            
            output_text = "\n".join(output_lines)

        alpaca_dataset.append({
            "instruction": instruction,
            "input": input_text,
            "output": output_text
        })

    with open(output_file, 'w', encoding='utf-8') as f: 
        json.dump(alpaca_dataset, f, indent=4, ensure_ascii=False)

    print(f"   Success! Generated {len(alpaca_dataset)} QA pairs from leftovers.")
    print(f"   -> Saved JSON to: {output_file}\n")


if __name__ == "__main__":
    TARGET_DIR = "data_finetuning"
    os.makedirs(TARGET_DIR, exist_ok=True)

    # 1. Process TRAIN dataset
    generate_leftovers_alpaca_dataset(
        input_file="data_generation_output/management_cases/DDI_Leftovers_Phase2_TRAIN.json",
        output_file=os.path.join(TARGET_DIR, "leftovers_cases_TRAIN.json"),
        dataset_name="TRAIN"
    )
    
    # 2. Process DEV dataset
    generate_leftovers_alpaca_dataset(
        input_file="data_generation_output/management_cases/DDI_Leftovers_Phase2_DEV.json",
        output_file=os.path.join(TARGET_DIR, "leftovers_cases_DEV.json"),
        dataset_name="DEV"
    )

    # 3. Batch convert generated JSONs to JSONL
    files_to_convert = [
        "leftovers_cases_TRAIN.json",
        "leftovers_cases_DEV.json"
    ]
    #convert_json_to_jsonl(TARGET_DIR, files_to_convert)

--- [START] Generating Realistic QA Dataset for TRAIN (Leftovers) ---
   Success! Generated 101 QA pairs from leftovers.
   -> Saved JSON to: data_finetuning\leftovers_cases_TRAIN.json

--- [START] Generating Realistic QA Dataset for DEV (Leftovers) ---
   Success! Generated 16 QA pairs from leftovers.
   -> Saved JSON to: data_finetuning\leftovers_cases_DEV.json



#### **2. TEST Dataset**

##### **2.1 QA generation**

This script processes the leftovers from the other steps and generate an evaluation dataset. The generated QA pairs are partitioned into two files:
* **Open-Ended QA (`_open.json`)**
* **Multiple Choice Questions (`_mcq.json`)**

**Key Logic**
* **Clinical Logic:** Assigns the targeted replacement drug as the "proposed addition" to teach the model to reject unsafe new prescriptions rather than disrupt current medications.
* **Formatting:** It merges overlapping directives, cleans shorthands (e.g., `; monitor`), and injects verbs (e.g., `"Consider substitute: "`) to prevent fragmented phrasing.

In [13]:
import json
import random
import re
import os

random.seed(42)

INPUT_FILE = os.path.join("data_generation_output", "management_cases", "DDI_Leftovers_Phase2_TEST.json")
OUTPUT_DIR = "data_finetuning"

INSTRUCTIONS_OPEN_QA = [
    "Offer clear, professional clinical support to the treating physician. Analyze the pharmacological interaction between the specified active substances.",
    "You are a specialized clinical AI. Provide precise, actionable medical insights to the attending physician. Evaluate the clinical safety of combining these two drugs.",
    "You are a specialized AI assistant for healthcare. Supply reliable, evidence-based support directly to the physician. Identify any drug-drug interactions for this combination and recommend clinical adjustments.",
    "Act as an expert clinical assistant. Deliver brief, evidence-based recommendations to the healthcare provider. Provide a pharmacological safety assessment for this drug pairing."
]

# For targets A or B (current vs Proposed)
INPUT_TEMPLATES_ASYMMETRIC = [
    "Is it safe to prescribe {proposed_drug} to a patient already taking {current_drug}?",
    "I'm considering adding {proposed_drug} to a regimen that includes {current_drug}. Are there any contraindications?",
    "Patient is currently on {current_drug}. Can I start them on {proposed_drug}?"
]

# For targets BOTH or N (General combination)
INPUT_TEMPLATES_SYMMETRIC = [
    "Evaluate the interaction risk of co-administering {current_drug} and {proposed_drug}.",
    "Review the clinical safety of combining {current_drug} and {proposed_drug}.",
    "Are there any pharmacological concerns with the concurrent use of {current_drug} and {proposed_drug}?"
]

INSTRUCTION_MCQ = "Read the question and options carefully. Output ONLY the letter (A, B, C, or D) corresponding to the correct answer."

DISCLAIMER = "Clinical note: The significance of drug–drug interactions may vary depending on individual patient factors and specific drug characteristics, including dosage. A thorough clinical evaluation is recommended before any therapeutic adjustments."

# --- FORMATTING HELPERS ---

def clean_clinical_text(text):
    if not text:
        return ""
    text = text.strip()
    
    text = text[0].upper() + text[1:]
    if not text.endswith('.'):
        text += '.'
        
    return text

def merge_action_effect(effect_text, action_text):
    # Helper function to merge redundant clinical directives
    if not effect_text:
        return action_text
        
    e_clean = effect_text.strip().rstrip('.')
    a_clean = action_text.strip().rstrip('.')
    
    e_lower = e_clean.lower()
    a_lower = a_clean.lower()
    
    # Create a clean 'core' of our own prefix so as not to break the redundancy check
    a_core = a_lower.replace("consider substitute: ", "").strip()
    e_core = e_lower.replace("consider substitute: ", "").strip()
    
    # CASE 1: Both start with "Monitor"
    if e_lower.startswith("monitor") and a_lower.startswith("monitor"):
        a_remainder = a_clean[len("Monitor"):].strip() 
        if a_remainder.startswith("or ") or a_remainder.startswith("and "):
            return f"{e_clean} {a_remainder}."
        else:
            # CREATE BULLET POINTS FOR INDEPENDENT SENTENCES
            return f"\n- {e_clean}.\n- {a_clean}."
            
    # CASE 2: Both involve "Space"
    elif "space" in e_lower and "space" in a_lower:
        time_match = re.search(r'\b(by\s+.*?)$', e_clean, re.IGNORECASE)
        if time_match:
            time_part = time_match.group(1)
            return f"Space doses appropriately ({time_part})."
        else:
            e_remainder = re.sub(r'(?i)^space\s+(doses\s+)?', '', e_clean).strip()
            if e_remainder:
                if e_remainder.startswith("or ") or e_remainder.startswith("and "):
                    return f"Space doses appropriately {e_remainder}."
                else:
                    return f"Space doses appropriately and {e_remainder}."
            else:
                return f"Space doses appropriately."
                
    # CASE 3: Complete redundancy ("Consider substitute: ...")
    elif a_core in e_core:
        return f"{e_clean}."
    elif e_core in a_core:
        return f"{a_clean}."
            
    # CASE 4: Different verbs (CREATE BULLET POINTS)
    else:
        return f"\n- {e_clean}.\n- {a_clean}."


# --- MAIN GENERATION FUNCTION ---

def generate_leftovers_test_dataset():
    print("--- [START] Generating Test Dataset for Leftovers ---")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if not os.path.exists(INPUT_FILE):
        print(f"   [ERROR] Required file not found: {INPUT_FILE}")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if not data:
        print(f"   [ERROR] File is empty: {INPUT_FILE}")
        return

    # =========================================================
    # 1. BUILD GLOBAL POOLS FOR DISTRACTORS
    # =========================================================
    global_diagnosis_pool = []
    global_rec_pool = []
    
    for item in data:
        # For Task 3 (Recommendations) - apply merge logic to distractors
        effect_pool = clean_clinical_text(item.get("effect", ""))
        rec_pool = clean_clinical_text(item.get("Safer_alternative", "").strip())
        target_pool = item.get("replacement_target", "").upper()
        
        if re.search(r';\s*monitor\.*$', effect_pool, re.IGNORECASE):
            effect_pool = re.sub(r';\s*monitor\.*$', '', effect_pool, flags=re.IGNORECASE).strip()
            effect_pool = clean_clinical_text(effect_pool)
            
            rec_lower = rec_pool.lower()
            action_keywords = ("monitor", "space", "avoid", "use", "discontinue", "maintain", "adjust", "start", "consider", "other", "alternative", "loop", "withhold", "reduce")
            if target_pool in ["A", "B"] and not rec_lower.startswith(action_keywords):
                rec_pool = f"Consider substitute: {rec_pool}"
            
            rec_pool = f"\n- Monitor.\n- {rec_pool}"

        is_action_effect_pool = effect_pool.lower().startswith(("monitor", "space", "maintain"))
        
        if is_action_effect_pool and target_pool in ["A", "B"]:
            rec_lower = rec_pool.lower()
            action_keywords = ("monitor", "space", "avoid", "use", "discontinue", "maintain", "adjust", "start", "consider", "other", "alternative", "loop", "withhold", "reduce")
            if not rec_lower.startswith(action_keywords):
                rec_pool = f"Consider substitute: {rec_pool}"

        if is_action_effect_pool:
            raw_mcq = merge_action_effect(effect_pool, rec_pool)
        else:
            raw_mcq = rec_pool
            
        # Flatten bullets into a single string for MCQ options (e.g., "- Monitor.\n- Avoid" -> "Monitor. Avoid")
        mcq_recommendation = raw_mcq.replace("\n- ", " ").strip()
        
        if mcq_recommendation and mcq_recommendation not in global_rec_pool:
            global_rec_pool.append(mcq_recommendation)
        

    test_dataset_open = []
    test_dataset_mcq = []

    # =========================================================
    # 2. MAIN LOOP: GENERATE QA AND MCQ PAIRS
    # =========================================================
    for item in data:
        drug_a = item.get("drug_a", "").strip()
        drug_b = item.get("drug_b", "").strip()
        
        severity = item.get("severity_label", item.get("severity", "Unknown")).upper()
        
        mechanism = clean_clinical_text(item.get("mechanism", ""))
        effect_raw = clean_clinical_text(item.get("effect", ""))
        rationale = clean_clinical_text(item.get("rationale", ""))
        alt_text = item.get("Safer_alternative", "").strip()
        target = item.get("replacement_target", "").upper()

        recommendation = clean_clinical_text(alt_text)

        # --- CLINICAL LOGIC ---
        if target == "A":
            current_drug = drug_b
            proposed_drug = drug_a
        elif target == "B":
            current_drug = drug_a
            proposed_drug = drug_b
        else:
            if random.choice([True, False]):
                current_drug = drug_a
                proposed_drug = drug_b
            else:
                current_drug = drug_b
                proposed_drug = drug_a

        # =========================================================
        # TASK 1: OPEN-ENDED QA 
        # =========================================================
        instruction_open = random.choice(INSTRUCTIONS_OPEN_QA)
        
        if target in ["A", "B"]:
            input_template = random.choice(INPUT_TEMPLATES_ASYMMETRIC)
        else:
            input_template = random.choice(INPUT_TEMPLATES_SYMMETRIC)
            
        input_text_open = input_template.format(current_drug=current_drug, proposed_drug=proposed_drug)

        if "none needed" in recommendation.lower():
            output_text_open = f"Co-administration of {drug_a} and {drug_b} is considered generally safe. No significant interaction expected."
        else:
            rec_label = "**Recommendations**"

            header = f"Clinical attention is required for the co-administration of {drug_a} and {drug_b}."
            
            output_lines = [
                header + "\n",
                f"Interaction details between {drug_a} and {drug_b}:",
                f"**Severity:** {severity}",
                f"**Mechanism:** {mechanism}"
            ]
            
            # --- MERGE LOGIC FOR OPEN QA ---
            if re.search(r';\s*monitor\.*$', effect_raw, re.IGNORECASE):
                effect_raw = re.sub(r';\s*monitor\.*$', '', effect_raw, flags=re.IGNORECASE).strip()
                effect_raw = clean_clinical_text(effect_raw)
                
                rec_lower = recommendation.lower()
                action_keywords = ("monitor", "space", "avoid", "use", "discontinue", "maintain", "adjust", "start", "consider", "other", "alternative", "loop", "withhold", "reduce")
                if target in ["A", "B"] and not rec_lower.startswith(action_keywords):
                    recommendation = f"Consider substitute: {recommendation}"
                
                recommendation = f"\n- Monitor.\n- {recommendation}"

            is_action_effect = effect_raw.lower().startswith(("monitor", "space", "maintain"))
            
            if is_action_effect and target in ["A", "B"]:
                rec_lower = recommendation.lower()
                action_keywords = ("monitor", "space", "avoid", "use", "discontinue", "maintain", "adjust", "start", "consider", "other", "alternative", "loop", "withhold", "reduce")
                if not rec_lower.startswith(action_keywords):
                    recommendation = f"Consider substitute: {recommendation}"

            if not is_action_effect and effect_raw:
                output_lines.append(f"**Effect:** {effect_raw}")
                if recommendation.startswith("\n"):
                    output_lines.append(f"{rec_label}:{recommendation}")
                else:
                    output_lines.append(f"{rec_label}: {recommendation}")
            elif is_action_effect:
                merged_recommendation = merge_action_effect(effect_raw, recommendation)
                if merged_recommendation.startswith("\n"):
                    output_lines.append(f"{rec_label}:{merged_recommendation}")
                else:
                    output_lines.append(f"{rec_label}: {merged_recommendation}")
            else:
                output_lines.append(f"{rec_label}: {recommendation}")

            output_lines.append(f"**Rationale:** {rationale}")
            output_lines.append(f"\n*{DISCLAIMER}*")
            output_text_open = "\n".join(output_lines)

        test_dataset_open.append({
            "instruction": instruction_open,
            "input": input_text_open,
            "output": output_text_open
        })


        # =========================================================
        # TASK 2: MCQ (Severity Classification)
        # =========================================================
        letters = ['A', 'B', 'C', 'D']
        all_severities = ["MAJOR", "MODERATE", "MINOR", "NO INTERACTION"] 
        
        correct_severity = severity.upper()
        if correct_severity not in all_severities:
            all_severities.append(correct_severity)

        wrong_severities = [s for s in all_severities if s != correct_severity]
        wrong_diag_options = random.sample(wrong_severities, min(3, len(wrong_severities)))

        diag_options = [correct_severity] + wrong_diag_options
        random.shuffle(diag_options)
        
        correct_letter_diag = letters[diag_options.index(correct_severity)]

        diag_options_text = ""
        for idx, opt in enumerate(diag_options):
            diag_options_text += f"{letters[idx]}. {opt}\n"

        # --- MCQ QUESTION ---
        if target in ["A", "B"]:
            mcq_stem = f"A patient is currently taking {current_drug}. I'm considering prescribing {proposed_drug}."
        else:
            mcq_stem = f"Consider a patient taking {current_drug} and {proposed_drug} simultaneously."

        input_text_diag = (
            f"###Question: {mcq_stem} "
            f"Which of the following is the severity level associated with this drug combination?\n"
            f"###Options:\n{diag_options_text.strip()}"
        )

        test_dataset_mcq.append({
            "instruction": INSTRUCTION_MCQ,
            "input": input_text_diag,
            "output": correct_letter_diag,
            "topic": "Severity_Classification"
        })


        # =========================================================
        # TASK 3: MedQA MCQ (Recommendation / Action)
        # =========================================================
        # Use the fully processed recommendation (either merged or standard)
        if is_action_effect:
            raw_mcq_rec = merge_action_effect(effect_raw, recommendation)
        else:
            raw_mcq_rec = recommendation

        # Flatten bullet points into a single readable string 
        final_mcq_rec = raw_mcq_rec.replace("\n- ", " ").strip()

        wrong_rec_options = []
        random.shuffle(global_rec_pool)
        for rec in global_rec_pool:
            if rec != final_mcq_rec and rec not in wrong_rec_options:
                wrong_rec_options.append(rec)
            if len(wrong_rec_options) == 3:
                break

        rec_options = [final_mcq_rec] + wrong_rec_options
        random.shuffle(rec_options)

        options_text_rec = ""
        correct_letter_rec = ""

        for idx, opt in enumerate(rec_options):
            options_text_rec += f"{letters[idx]}. {opt}\n"
            if opt == final_mcq_rec:
                correct_letter_rec = letters[idx]

        input_text_rec = (
            f"###Question: {mcq_stem} "
            f"Which of the following is the most appropriate clinical recommendation?\n"
            f"###Options:\n{options_text_rec.strip()}"
        )

        test_dataset_mcq.append({
            "instruction": INSTRUCTION_MCQ,
            "input": input_text_rec,
            "output": correct_letter_rec,
            "topic": "Clinical_Management"
        })

    out_open = "leftovers_cases_TEST_open.json"
    out_mcq = "leftovers_cases_TEST_mcq.json"
    
    with open(os.path.join(OUTPUT_DIR, out_open), 'w', encoding='utf-8') as f:
        json.dump(test_dataset_open, f, indent=4, ensure_ascii=False)
        
    with open(os.path.join(OUTPUT_DIR, out_mcq), 'w', encoding='utf-8') as f:
        json.dump(test_dataset_mcq, f, indent=4, ensure_ascii=False)

    print(f"   Success! Generated {len(test_dataset_open)} Open QA pairs and {len(test_dataset_mcq)} MCQ pairs (Tasks 2 & 3).")
    print(f"   -> Saved to: {OUTPUT_DIR}\n")


if __name__ == "__main__":
    generate_leftovers_test_dataset()

--- [START] Generating Test Dataset for Leftovers ---
   Success! Generated 14 Open QA pairs and 28 MCQ pairs (Tasks 2 & 3).
   -> Saved to: data_finetuning



### **Final Merge**

This script agregates the diverse scenarios: Complete Cases, Management Cases, and Leftovers.

- **Data Aggregation:** Merges the isolated JSON outputs from previous pipeline stages into definitive training and evaluation splits: `final_TRAIN`, `final_DEV`, `final_TEST_open`, and `final_TEST_mcq`. 
- **Shuffling the data**
- **JSONL format:** Writes the final datasets into the `.jsonl` format.

In [14]:
import json
import os
import random

random.seed(42)

INPUT_DIR = "data_finetuning"
OUTPUT_DIR = os.path.join(INPUT_DIR, "final_datasets")

# Map the exact generated filenames to their respective final splits
DATASET_SPLITS = {
    "final_TRAIN": [
        "complete_cases_TRAIN.json",
        "management_cases_TRAIN.json",
        "leftovers_cases_TRAIN.json"
    ],
    "final_DEV": [
        "complete_cases_DEV.json",
        "management_cases_DEV.json",
        "leftovers_cases_DEV.json"
    ],
    "final_TEST_open": [
        "complete_cases_TEST_open.json",
        "management_cases_TEST_open.json",
        "leftovers_cases_TEST_open.json"
    ],
    "final_TEST_mcq": [
        "complete_cases_TEST_mcq.json",
        "management_cases_TEST_mcq.json",
        "leftovers_cases_TEST_mcq.json"
    ]
}

# --- HELPER FUNCTION ---
def merge_and_export(split_name, file_list):
    print(f"--- [START] Merging files for {split_name} ---")
    
    combined_data = []
    missing_files = 0
    
    # Read and append data from all source files
    for filename in file_list:
        filepath = os.path.join(INPUT_DIR, filename)
        
        if not os.path.exists(filepath):
            print(f"   [WARNING] File not found: {filename}")
            missing_files += 1
            continue
            
        with open(filepath, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
                combined_data.extend(data)
                print(f"   -> Loaded {len(data)} items from {filename}")
            except json.JSONDecodeError:
                print(f"   [ERROR] Could not decode JSON from {filename}")

    if not combined_data:
        print(f"   [SKIP] No data gathered for {split_name}.\n")
        return

    # Shuffle the combined dataset
    random.shuffle(combined_data)
    
    # Export to .jsonl
    jsonl_filename = f"{split_name}.jsonl"
    jsonl_filepath = os.path.join(OUTPUT_DIR, jsonl_filename)
    
    with open(jsonl_filepath, 'w', encoding='utf-8') as f_out:
        for item in combined_data:
            f_out.write(json.dumps(item, ensure_ascii=False) + '\n')
            
    print(f"   [SUCCESS] {split_name} compiled.")
    print(f"   -> Total QA pairs: {len(combined_data)}")
    print(f"   -> Saved to: {jsonl_filepath}\n")


# --- MAIN EXECUTION ---
if __name__ == "__main__":
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    for split_name, files in DATASET_SPLITS.items():
        merge_and_export(split_name, files)
        
    print(f"Final datasets are in: {OUTPUT_DIR}")

--- [START] Merging files for final_TRAIN ---
   -> Loaded 20 items from complete_cases_TRAIN.json
   -> Loaded 16 items from management_cases_TRAIN.json
   -> Loaded 101 items from leftovers_cases_TRAIN.json
   [SUCCESS] final_TRAIN compiled.
   -> Total QA pairs: 137
   -> Saved to: data_finetuning\final_datasets\final_TRAIN.jsonl

--- [START] Merging files for final_DEV ---
   -> Loaded 1 items from complete_cases_DEV.json
   [WARNING] File not found: management_cases_DEV.json
   -> Loaded 16 items from leftovers_cases_DEV.json
   [SUCCESS] final_DEV compiled.
   -> Total QA pairs: 17
   -> Saved to: data_finetuning\final_datasets\final_DEV.jsonl

--- [START] Merging files for final_TEST_open ---
   -> Loaded 2 items from complete_cases_TEST_open.json
   -> Loaded 2 items from management_cases_TEST_open.json
   -> Loaded 14 items from leftovers_cases_TEST_open.json
   [SUCCESS] final_TEST_open compiled.
   -> Total QA pairs: 18
   -> Saved to: data_finetuning\final_datasets\final_TE

### **Few-shot prompting**

A 5-shot example set consisting of two complete cases, two management cases, and one leftover case was derived from the training split for few-shot evaluation.

#### **Complete cases**

In [15]:
import json
import random
import re
import os

random.seed(42)

INPUT_FILE = os.path.join("data_generation_output", "complete_cases", "complete_cases_TRAIN.json")
MEDS_FILE = os.path.join("..", "Medicines", "datasets_split", "dataset_meds_TRAIN_v2.json")
OUTPUT_DIR = os.path.join("data_finetuning", "final_datasets")
OUTPUT_FEWSHOT_FILE = os.path.join(OUTPUT_DIR, "few_shot_examples_ddi_complete.jsonl")

# --- REGEX PATTERNS ---
SYSTEMIC_PATTERN = re.compile(r'\b(tablet|capsule)\b', re.IGNORECASE)
TOPICAL_PATTERN = re.compile(r'\b(cream|gel|ointment|spray|drops?|lotion|shampoo|topical|soap|syrup|suppository)\b', re.IGNORECASE)
INSTRUCTION_MCQ = "Read the question and options carefully. Output ONLY the letter (A, B, C, or D) corresponding to the correct answer."

# --- FORMATTING HELPERS ---
def strip_dosage(composition_text):
    return re.sub(r'\s*\(.*?\)', '', composition_text).strip()

def get_composition_str(med_dict):
    comp_list = med_dict.get('Composition', [])
    return ", ".join(comp_list) if comp_list else "Unknown"

def smart_lower(text):
    if not text: return ""
    parts = re.split(r'(\(.*?\))', str(text))
    result = ""
    for part in parts:
        if part.startswith('(') and part.endswith(')'): result += part 
        else: result += part.lower()  
    return result.strip()

# --- MAIN FUNCTION ---
def generate_fewshot_ddi_dataset():
    print("--- [START] Generating 2-Shot Examples (Complete Cases TRAIN) ---")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if not os.path.exists(INPUT_FILE) or not os.path.exists(MEDS_FILE):
        print(" [ERROR] Required TRAIN files not found. Check your paths.")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as f: grouped_data = json.load(f)
    with open(MEDS_FILE, 'r', encoding='utf-8') as f: meds_data = json.load(f)
        
    all_meds_list = list(meds_data.values()) if isinstance(meds_data, dict) else meds_data

    severity_questions = []
    substitution_questions = []
    
    for group in grouped_data:
        severity = group.get("severity", "Unknown").upper()
        cases = group.get("generated_cases", [])
        if not cases: continue 

        def score_medication(med_name):
            if TOPICAL_PATTERN.search(med_name): return -10 
            elif SYSTEMIC_PATTERN.search(med_name): return 2 
            else: return 1 

        def score_case(c):
            return score_medication(c["medication_A"]["medicine_name"]) + score_medication(c["medication_B"]["medicine_name"])

        cases.sort(key=score_case, reverse=True)
        best_case = cases[0] 

        med_A = best_case["medication_A"]
        med_B = best_case["medication_B"]
        sols = best_case["solutions"]
        target = sols["target_replaced"]

        if target == "A":
            current_med, proposed_med = med_B, med_A
        else:
            current_med, proposed_med = med_A, med_B

        cond_curr = smart_lower(str(current_med.get('condition', '')))
        cond_prop = smart_lower(str(proposed_med.get('condition', '')))
        drug_curr_clean = strip_dosage(current_med['composition'])
        drug_prop_clean = strip_dosage(proposed_med['composition'])
        letters = ['A', 'B', 'C', 'D']

        # MCQ 1: SEVERITY
        all_severities = ["MAJOR", "MODERATE", "MINOR", "NO INTERACTION"] 
        correct_severity = severity.upper()
        if correct_severity not in all_severities: all_severities.append(correct_severity)

        wrong_severities = [s for s in all_severities if s != correct_severity]
        diag_options = [correct_severity] + random.sample(wrong_severities, min(3, len(wrong_severities)))
        random.shuffle(diag_options)
        
        correct_letter_diag = letters[diag_options.index(correct_severity)]
        diag_options_text = "".join([f"{letters[idx]}. {opt}\n" for idx, opt in enumerate(diag_options)])

        severity_questions.append({
            "instruction": INSTRUCTION_MCQ,
            "input": f"###Question: A patient is taking {drug_curr_clean} [{current_med['medicine_name']}] for {cond_curr}. I'm planning to prescribe {drug_prop_clean} [{proposed_med['medicine_name']}] for {cond_prop}. Which of the following is the severity level associated with this drug combination?\n###Options:\n{diag_options_text.strip()}",
            "output": correct_letter_diag
        })

        # MCQ 2: SUBSTITUTION
        target_cond = smart_lower(sols.get("condition_treated", ""))
        if sols.get("alternative_meds"):
            correct_alt_med = sols["alternative_meds"][0]
            clean_correct_comp = strip_dosage(correct_alt_med["composition"])
            wrong_options = []
            random.shuffle(all_meds_list) 

            for dist_med in all_meds_list:
                dist_uses_list = [smart_lower(u) for u in dist_med.get('Uses', [])]
                dist_comp = strip_dosage(get_composition_str(dist_med))
                has_semantic_overlap = any(target_cond in u or u in target_cond for u in dist_uses_list)
                
                if not has_semantic_overlap and dist_comp != clean_correct_comp:
                    wrong_options.append({"medicine_name": dist_med["Medicine Name"], "composition": dist_comp})
                if len(wrong_options) == 3: break

            if len(wrong_options) == 3:
                mcq_options = [{"medicine_name": correct_alt_med["medicine_name"], "composition": clean_correct_comp}] + wrong_options
                random.shuffle(mcq_options)
                
                subs_options_text = "".join([f"{letters[idx]}. {opt['medicine_name']} ({opt['composition']})\n" for idx, opt in enumerate(mcq_options)])
                correct_letter_subs = letters[mcq_options.index({"medicine_name": correct_alt_med["medicine_name"], "composition": clean_correct_comp})]

                substitution_questions.append({
                    "instruction": INSTRUCTION_MCQ,
                    "input": f"###Question: Patient taking {drug_curr_clean} [{current_med['medicine_name']}] requires medication for {target_cond}. {drug_prop_clean} [{proposed_med['medicine_name']}] is contraindicated due to a {severity.lower()} drug-drug interaction. Which of the following is the safest alternative to substitute the contraindicated medication?\n###Options:\n{subs_options_text.strip()}",
                    "output": correct_letter_subs
                })

    print(f"\n[INFO] Found {len(severity_questions)} severity and {len(substitution_questions)} substitution questions.")
    
    if not severity_questions or not substitution_questions:
        print("[ERROR] Not enough valid questions.")
        return

    final_few_shot = [random.choice(severity_questions), random.choice(substitution_questions)]
    random.shuffle(final_few_shot)

    with open(OUTPUT_FEWSHOT_FILE, 'w', encoding='utf-8') as f:
        for item in final_few_shot: f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print(f" -> Success! Generated 2 Complete Cases Examples.\n -> Saved to: {OUTPUT_FEWSHOT_FILE}\n")

if __name__ == "__main__":
    generate_fewshot_ddi_dataset()

--- [START] Generating 2-Shot Examples (Complete Cases TRAIN) ---

[INFO] Found 20 severity and 18 substitution questions.
 -> Success! Generated 2 Complete Cases Examples.
 -> Saved to: data_finetuning\final_datasets\few_shot_examples_ddi_complete.jsonl



#### **Management cases**

In [16]:
import json
import os
import random
import re
import itertools

random.seed(42)

INPUT_DIR = os.path.join("data_generation_output", "complete_cases")
MEDS_INPUT_DIR = os.path.join("..", "Medicines", "datasets_split")
LEFTOVERS_IN = "DDI_Leftovers_TRAIN.json"
MEDS_IN = "dataset_meds_TRAIN_v2.json"

OUTPUT_DIR = os.path.join("data_finetuning", "final_datasets")
OUTPUT_FEWSHOT_FILE = os.path.join(OUTPUT_DIR, "few_shot_examples_ddi_complete.jsonl")

# --- ALIGNMENT WITH TEST SET ---
MANAGEMENT_PHRASES_TEST = {
    "space doses appropriately",
    "monitor or continue with caution",
    "none needed",
    "monitor or adjust doses"
}

# --- REGEX PATTERNS ---
SYSTEMIC_PATTERN = re.compile(r'\b(tablet|capsule)\b', re.IGNORECASE)
TOPICAL_PATTERN = re.compile(r'\b(cream|gel|ointment|spray|drops?|lotion|shampoo|topical|soap|syrup|suppository)\b', re.IGNORECASE)
INSTRUCTION_MCQ = "Read the question and options carefully. Output ONLY the letter (A, B, C, or D) corresponding to the correct answer."

# --- FORMATTING HELPERS ---
def clean_clinical_text(text):
    if not text: return ""
    text = text.strip()
    text = text[0].upper() + text[1:]
    return text + '.' if not text.endswith('.') else text

def clean_composition_text(text):
    return re.sub(r'\s*\(.*?\)', '', text).strip().lower()

def strip_dosage(comp):
    return re.sub(r'\s*\(.*?\)', '', comp).strip()

def get_composition_str(med_dict):
    c = med_dict.get('Composition', [])
    return ", ".join(c) if c else "Unknown"

def smart_lower(text):
    if not text: return ""
    return "".join(p if p.startswith('(') and p.endswith(')') else p.lower() for p in re.split(r'(\(.*?\))', str(text))).strip()

def merge_action_effect(e_text, a_text):
    if not e_text: return a_text
    e, a = e_text.strip().rstrip('.'), a_text.strip().rstrip('.')
    el, al = e.lower(), a.lower()
    
    if el.startswith("monitor") and al.startswith("monitor"):
        arem = a[len("Monitor"):].strip() 
        return f"{e} {arem}." if arem.startswith(("or ", "and ")) else f"{e}. {a}."
    elif "space" in el and "space" in al:
        tm = re.search(r'\b(by\s+.*?)$', e, re.IGNORECASE)
        if tm: return f"Space doses appropriately ({tm.group(1)})."
        erem = re.sub(r'(?i)^space\s+(doses\s+)?', '', e).strip()
        if erem: return f"Space doses appropriately {erem}." if erem.startswith(("or ", "and ")) else f"Space doses appropriately and {erem}."
        return "Space doses appropriately."
    return f"{e}. {a}."

def build_medicine_index(meds_data):
    idx = {}
    mlist = meds_data.values() if isinstance(meds_data, dict) else meds_data
    for m in mlist:
        c = m.get('Composition', [])
        if len(c) == 1:
            ck = clean_composition_text(c[0])
            if ck not in idx: idx[ck] = []
            idx[ck].append(m)
    return idx

def get_unique_uses(src, cmp):
    cmp_uses = " ".join(cmp.get('Uses', [])).lower()
    return [u for u in src.get('Uses', []) if u.lower() not in cmp_uses]

# --- MAIN FUNCTION ---
def generate_management_fewshot():
    print(f"\n--- [START] Generating 2-Shot Examples (Management TRAIN -> Aligned to TEST) ---")
    
    leftover_path = os.path.join(INPUT_DIR, LEFTOVERS_IN)
    meds_path = os.path.join(MEDS_INPUT_DIR, MEDS_IN)

    if not os.path.exists(leftover_path) or not os.path.exists(meds_path):
        print(f" [ERROR] Required files not found.")
        return

    with open(leftover_path, 'r', encoding='utf-8') as f: leftovers = json.load(f)
    with open(meds_path, 'r', encoding='utf-8') as f: meds_data = json.load(f)
    
    med_index = build_medicine_index(meds_data)
    
    severity_questions, action_questions, global_rec_pool, grouped_output = [], [], [], []

    # 1. First Pass: Build the strict global pool of recommendations
    for item in leftovers:
        alt_text = item.get("Safer_alternative", "").strip()
        if alt_text.lower() not in MANAGEMENT_PHRASES_TEST: continue # STRICT ALIGNMENT FILTER

        e_raw = clean_clinical_text(item.get('effect'))
        a_fmt = clean_clinical_text(alt_text)
        is_ae = e_raw.lower().startswith(("monitor", "space"))
        
        mrec = merge_action_effect(e_raw, a_fmt) if is_ae else a_fmt
        if mrec not in global_rec_pool: global_rec_pool.append(mrec)

    # 2. Second Pass: Generate valid cases
    for item in leftovers:
        alt_text = item.get("Safer_alternative", "").strip()
        if alt_text.lower() not in MANAGEMENT_PHRASES_TEST: continue 

        ma = med_index.get(item.get('drug_a', '').strip().lower())
        mb = med_index.get(item.get('drug_b', '').strip().lower())
        if not ma or not mb: continue 

        valid_scenarios = []
        ab_pairs = list(itertools.product(ma, mb))
        random.shuffle(ab_pairs) 

        for a, b in ab_pairs:
            if not a.get('Uses') or not b.get('Uses'): continue
            vuA, vuB = get_unique_uses(a, b), get_unique_uses(b, a)
            if vuA and vuB:
                valid_scenarios.append({
                    "patient_context": {"medication": a['Medicine Name'], "composition": get_composition_str(a), "condition": random.choice(vuA)},
                    "proposed_context": {"medication": b['Medicine Name'], "composition": get_composition_str(b), "condition": random.choice(vuB)}
                })

        if valid_scenarios:
            grouped_output.append({
                "severity": item.get('severity_label', 'Unknown').upper(),
                "effect": item.get('effect'),
                "recommendation": alt_text,
                "generated_cases": valid_scenarios
            })

    # 3. Third Pass: Build MCQs
    for group in grouped_output:
        cases = group["generated_cases"]
        def score_case(c): return (2 if SYSTEMIC_PATTERN.search(c["patient_context"]["medication"]) else -10 if TOPICAL_PATTERN.search(c["patient_context"]["medication"]) else 1) + (2 if SYSTEMIC_PATTERN.search(c["proposed_context"]["medication"]) else -10 if TOPICAL_PATTERN.search(c["proposed_context"]["medication"]) else 1)
        cases.sort(key=score_case, reverse=True)
        c1 = cases[0]
        
        mA, mB = c1["patient_context"], c1["proposed_context"]
        dac, dbc = strip_dosage(mA["composition"]), strip_dosage(mB["composition"])
        cal, cbl = smart_lower(mA.get("condition", "")), smart_lower(mB.get("condition", ""))
        letters = ['A', 'B', 'C', 'D']

        # Task 2: Severity
        sev = group["severity"].upper()
        all_sev = list(set(["MAJOR", "MODERATE", "MINOR", "NO INTERACTION"] + [sev]))
        diag_opts = [sev] + random.sample([s for s in all_sev if s != sev], 3)
        random.shuffle(diag_opts)
        
        severity_questions.append({
            "instruction": INSTRUCTION_MCQ,
            "input": f"###Question: A patient is taking {dac} [{mA['medication']}] for {cal}. I'm considering prescribing {dbc} [{mB['medication']}] for {cbl}. Which of the following is the severity level associated with this drug combination?\n###Options:\n" + "".join([f"{letters[i]}. {opt}\n" for i, opt in enumerate(diag_opts)]).strip(),
            "output": letters[diag_opts.index(sev)]
        })

        # Task 3: Management Action
        eraw, afmt = clean_clinical_text(group["effect"]), clean_clinical_text(group["recommendation"])
        crec = merge_action_effect(eraw, afmt) if eraw.lower().startswith(("monitor", "space")) else afmt

        wrec = []
        random.shuffle(global_rec_pool)
        for r in global_rec_pool:
            if r != crec and r not in wrec: wrec.append(r)
            if len(wrec) == 3: break

        # Only add question if we successfully found 3 unique distractors
        if len(wrec) == 3:
            ropts = [crec] + wrec
            random.shuffle(ropts)
            action_questions.append({
                "instruction": INSTRUCTION_MCQ,
                "input": f"###Question: A patient is taking {dac} [{mA['medication']}] for {cal}. I'm considering prescribing {dbc} [{mB['medication']}] for {cbl}. Which of the following is the most appropriate clinical management?\n###Options:\n" + "".join([f"{letters[i]}. {opt}\n" for i, opt in enumerate(ropts)]).strip(),
                "output": letters[ropts.index(crec)]
            })

    print(f"\n[INFO] Found {len(severity_questions)} severity and {len(action_questions)} action questions.")
    if not severity_questions or not action_questions: return print("[ERROR] Not enough valid questions.")

    final_few_shot = [random.choice(severity_questions), random.choice(action_questions)]
    random.shuffle(final_few_shot)

    # --- SAVE OUTPUT: APPEND MODE ---
    with open(OUTPUT_FEWSHOT_FILE, 'a', encoding='utf-8') as f:
        for item in final_few_shot: 
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print(f" -> Success! Appended 2 Management Examples to {OUTPUT_FEWSHOT_FILE}\n")

if __name__ == "__main__":
    generate_management_fewshot()


--- [START] Generating 2-Shot Examples (Management TRAIN -> Aligned to TEST) ---

[INFO] Found 14 severity and 14 action questions.
 -> Success! Appended 2 Management Examples to data_finetuning\final_datasets\few_shot_examples_ddi_complete.jsonl



#### **Leftovers**

In [17]:
import json
import random
import re
import os

random.seed(42)

INPUT_FILE = os.path.join("data_generation_output", "management_cases", "DDI_Leftovers_Phase2_TRAIN.json")

OUTPUT_DIR = os.path.join("data_finetuning", "final_datasets")
OUTPUT_FEWSHOT_FILE = os.path.join(OUTPUT_DIR, "few_shot_examples_ddi_complete.jsonl")

# --- REGEX & TEMPLATES ---
INSTRUCTION_MCQ = "Read the question and options carefully. Output ONLY the letter (A, B, C, or D) corresponding to the correct answer."

# --- FORMATTING HELPERS ---
def clean_clinical_text(text):
    if not text: return ""
    text = text.strip()
    text = text[0].upper() + text[1:]
    return text + '.' if not text.endswith('.') else text

def merge_action_effect(effect_text, action_text):
    if not effect_text: return action_text
        
    e_clean = effect_text.strip().rstrip('.')
    a_clean = action_text.strip().rstrip('.')
    e_lower = e_clean.lower()
    a_lower = a_clean.lower()
    
    a_core = a_lower.replace("consider substitute: ", "").strip()
    e_core = e_lower.replace("consider substitute: ", "").strip()
    
    if e_lower.startswith("monitor") and a_lower.startswith("monitor"):
        a_remainder = a_clean[len("Monitor"):].strip() 
        if a_remainder.startswith("or ") or a_remainder.startswith("and "):
            return f"{e_clean} {a_remainder}."
        else:
            return f"\n- {e_clean}.\n- {a_clean}."
            
    elif "space" in e_lower and "space" in a_lower:
        time_match = re.search(r'\b(by\s+.*?)$', e_clean, re.IGNORECASE)
        if time_match:
            return f"Space doses appropriately ({time_match.group(1)})."
        else:
            e_remainder = re.sub(r'(?i)^space\s+(doses\s+)?', '', e_clean).strip()
            if e_remainder:
                if e_remainder.startswith("or ") or e_remainder.startswith("and "):
                    return f"Space doses appropriately {e_remainder}."
                else:
                    return f"Space doses appropriately and {e_remainder}."
            else:
                return f"Space doses appropriately."
                
    elif a_core in e_core:
        return f"{e_clean}."
    elif e_core in a_core:
        return f"{a_clean}."
    else:
        return f"\n- {e_clean}.\n- {a_clean}."

# --- MAIN GENERATION FUNCTION ---
def generate_leftovers_fewshot():
    print("--- [START] Generating 1-Shot Example (Leftovers TRAIN) ---")
    
    if not os.path.exists(INPUT_FILE):
        print(f"   [ERROR] Required file not found: {INPUT_FILE}")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if not data:
        print(f"   [ERROR] File is empty: {INPUT_FILE}")
        return

    # 1. BUILD GLOBAL POOLS FOR DISTRACTORS
    global_rec_pool = []
    for item in data:
        effect_pool = clean_clinical_text(item.get("effect", ""))
        rec_pool = clean_clinical_text(item.get("Safer_alternative", "").strip())
        target_pool = item.get("replacement_target", "").upper()
        
        if re.search(r';\s*monitor\.*$', effect_pool, re.IGNORECASE):
            effect_pool = re.sub(r';\s*monitor\.*$', '', effect_pool, flags=re.IGNORECASE).strip()
            effect_pool = clean_clinical_text(effect_pool)
            
            rec_lower = rec_pool.lower()
            action_keywords = ("monitor", "space", "avoid", "use", "discontinue", "maintain", "adjust", "start", "consider", "other", "alternative", "loop", "withhold", "reduce")
            if target_pool in ["A", "B"] and not rec_lower.startswith(action_keywords):
                rec_pool = f"Consider substitute: {rec_pool}"
            
            rec_pool = f"\n- Monitor.\n- {rec_pool}"

        is_action_effect_pool = effect_pool.lower().startswith(("monitor", "space", "maintain"))
        
        if is_action_effect_pool and target_pool in ["A", "B"]:
            rec_lower = rec_pool.lower()
            action_keywords = ("monitor", "space", "avoid", "use", "discontinue", "maintain", "adjust", "start", "consider", "other", "alternative", "loop", "withhold", "reduce")
            if not rec_lower.startswith(action_keywords):
                rec_pool = f"Consider substitute: {rec_pool}"

        if is_action_effect_pool:
            raw_mcq = merge_action_effect(effect_pool, rec_pool)
        else:
            raw_mcq = rec_pool
            
        mcq_recommendation = raw_mcq.replace("\n- ", " ").strip()
        
        if mcq_recommendation and mcq_recommendation not in global_rec_pool:
            global_rec_pool.append(mcq_recommendation)
        
    all_leftover_questions = []

    # 2. MAIN LOOP: GENERATE MCQ PAIRS
    for item in data:
        drug_a = item.get("drug_a", "").strip()
        drug_b = item.get("drug_b", "").strip()
        severity = item.get("severity_label", item.get("severity", "Unknown")).upper()
        
        effect_raw = clean_clinical_text(item.get("effect", ""))
        alt_text = item.get("Safer_alternative", "").strip()
        target = item.get("replacement_target", "").upper()
        recommendation = clean_clinical_text(alt_text)

        # Inversion Logic
        if target == "A":
            current_drug, proposed_drug = drug_b, drug_a
        elif target == "B":
            current_drug, proposed_drug = drug_a, drug_b
        else:
            if random.choice([True, False]):
                current_drug, proposed_drug = drug_a, drug_b
            else:
                current_drug, proposed_drug = drug_b, drug_a

        #context
        if target in ["A", "B"]:
            mcq_stem = f"A patient is currently taking {current_drug}. I'm considering prescribing {proposed_drug}."
        else:
            mcq_stem = f"Consider a patient taking {current_drug} and {proposed_drug} simultaneously."

        letters = ['A', 'B', 'C', 'D']

        # TASK 2: Severity
        all_severities = ["MAJOR", "MODERATE", "MINOR", "NO INTERACTION"] 
        correct_severity = severity.upper()
        if correct_severity not in all_severities: all_severities.append(correct_severity)

        wrong_severities = [s for s in all_severities if s != correct_severity]
        diag_options = [correct_severity] + random.sample(wrong_severities, min(3, len(wrong_severities)))
        random.shuffle(diag_options)
        
        correct_letter_diag = letters[diag_options.index(correct_severity)]
        diag_options_text = "".join([f"{letters[idx]}. {opt}\n" for idx, opt in enumerate(diag_options)])

        input_text_diag = (
            f"###Question: {mcq_stem} Which of the following is the severity level associated with this drug combination?\n"
            f"###Options:\n{diag_options_text.strip()}"
        )

        all_leftover_questions.append({
            "instruction": INSTRUCTION_MCQ,
            "input": input_text_diag,
            "output": correct_letter_diag
        })

        # TASK 3: Recommendation
        is_action_effect = effect_raw.lower().startswith(("monitor", "space", "maintain"))
        if is_action_effect:
            raw_mcq_rec = merge_action_effect(effect_raw, recommendation)
        else:
            raw_mcq_rec = recommendation

        final_mcq_rec = raw_mcq_rec.replace("\n- ", " ").strip()

        wrong_rec_options = []
        random.shuffle(global_rec_pool)
        for rec in global_rec_pool:
            if rec != final_mcq_rec and rec not in wrong_rec_options:
                wrong_rec_options.append(rec)
            if len(wrong_rec_options) == 3: break

        if len(wrong_rec_options) == 3:
            rec_options = [final_mcq_rec] + wrong_rec_options
            random.shuffle(rec_options)

            options_text_rec = "".join([f"{letters[idx]}. {opt}\n" for idx, opt in enumerate(rec_options)])
            correct_letter_rec = letters[rec_options.index(final_mcq_rec)]

            input_text_rec = (
                f"###Question: {mcq_stem} Which of the following is the most appropriate clinical recommendation?\n"
                f"###Options:\n{options_text_rec.strip()}"
            )

            all_leftover_questions.append({
                "instruction": INSTRUCTION_MCQ,
                "input": input_text_rec,
                "output": correct_letter_rec
            })

    # --- FEW-SHOT SELECTION (Exactly 1 Question) ---
    print(f"\n[INFO] Generated {len(all_leftover_questions)} possible leftover questions.")
    
    if not all_leftover_questions:
        print("[ERROR] Not enough valid questions.")
        return

    # Pick exactly 1 question randomly to close the 5-shot loop
    final_few_shot = [random.choice(all_leftover_questions)]

    # --- SAVE OUTPUT: APPEND MODE ---
    with open(OUTPUT_FEWSHOT_FILE, 'a', encoding='utf-8') as f:
        for item in final_few_shot: 
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print(f" -> Success! Appended 1 Leftover Example to {OUTPUT_FEWSHOT_FILE}\n")

if __name__ == "__main__":
    generate_leftovers_fewshot()

--- [START] Generating 1-Shot Example (Leftovers TRAIN) ---

[INFO] Generated 202 possible leftover questions.
 -> Success! Appended 1 Leftover Example to data_finetuning\final_datasets\few_shot_examples_ddi_complete.jsonl

